# Leave-One-Camera-Out: Wavelength-Aware Spectral Tokenisation for Cross-Camera Hyperspectral Fruit-Ripeness and Wheat-Leaf Phenotyping, with a Leakage-Free Benchmark

**Purpose.** This notebook is the complete experimental pipeline for the paper. It builds a leakage-free
benchmark on two open datasets, trains a wavelength-aware spectral tokeniser (WST) and its baselines, and
computes every number, table and figure at run time.

| Stage | Section |
|---|---|
| Configuration, environment, hardware report | 1–2 |
| Data download and synthetic smoke-test data | 3 |
| Dataset indexing, schema discovery | 4 |
| Leakage-free splits with assertions | 5 |
| Cube loading, preprocessing, spectral resampling augmentation | 6 |
| Proposed model (WST) and masked-spectral pretraining | 7 |
| Baselines (neural and classical) | 8 |
| Metrics, training and evaluation engine | 9–10 |
| Experiment A: DeepHS Fruit leave-one-camera-out (+ ablations) | 11 |
| Experiment B: HyperLeaf2024 plot-level split and band-subset shift | 12 |
| Statistical tests | 13 |
| CPU versus GPU efficiency study | 14 |
| Tables and figures | 15 |
| Reproducibility appendix | 16 |

**How to run.**
1. Run everything once as it is (`RUN_MODE = "SMOKE_TEST"`). This generates small synthetic cubes in the
   same file formats as the real data and exercises every cell in about 10–15 minutes on a two-core CPU.
   The smoke test only proves that the pipeline runs: with four epochs on a few dozen cubes the networks
   stay near chance. To watch them learn on the synthetic data, raise `CFG["epochs"]` to about 80.
2. Download the real data (Section 3), confirm the annotation schema (Section 4), then set
   `RUN_MODE = "FULL_RUN"` and run again. Results are written to a separate folder.

**Integrity rules built into this notebook.**
* No result is hard-coded. Smoke-test numbers come from synthetic data, are stamped `SYNTHETIC`, are written
  to `results/smoke_test/` and **must never be reported in the paper**.
* Numbers quoted from the literature live only in `REFERENCE_NUMBERS` with their source, and are
  protocol-mismatched context, not comparisons.
* Items that could not be verified from primary sources are marked **NOT VERIFIED** where they occur.

## 1. Configuration

In [ ]:
from __future__ import annotations

import json
import math
import os
import platform
import random
import subprocess
import sys
import time
import urllib.request
import warnings
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd

# ----------------------------------------------------------------------------------------------
# Run mode. "SMOKE_TEST" = tiny synthetic data, few epochs.  "FULL_RUN" = real data, full protocol.
# ----------------------------------------------------------------------------------------------
RUN_MODE = os.environ.get("RUN_MODE", "SMOKE_TEST")
assert RUN_MODE in ("SMOKE_TEST", "FULL_RUN")
SMOKE = RUN_MODE == "SMOKE_TEST"
# Set USE_REAL_DATA=1 to smoke-test the pipeline on the real files with the short schedule.
USE_SYNTHETIC = SMOKE and os.environ.get("USE_REAL_DATA", "0") != "1"
DATA_SOURCE = "SYNTHETIC" if USE_SYNTHETIC else "REAL"

ROOT = Path(os.environ.get("PROJECT_ROOT", ".")).resolve()
DATA_DIR = ROOT / "data"
DEEPHS_DIR = DATA_DIR / "deephs_fruit"        # real DeepHS Fruit v2
HLEAF_DIR = DATA_DIR / "hyperleaf2024"        # real HyperLeaf2024
SYN_DIR = DATA_DIR / "synthetic"              # generated by this notebook
OUT_DIR = ROOT / "results" / ("smoke_test" if USE_SYNTHETIC else RUN_MODE.lower())
FIG_DIR = OUT_DIR / "figures"
for _p in (DATA_DIR, DEEPHS_DIR, HLEAF_DIR, SYN_DIR, OUT_DIR, FIG_DIR):
    _p.mkdir(parents=True, exist_ok=True)

SEEDS = [0, 1] if SMOKE else list(range(10))

CFG = dict(
    # ---- data ----
    fruits=["mango", "kiwi"] if SMOKE else ["avocado", "kiwi", "mango", "kaki", "papaya"],
    tasks=["ripeness"] if SMOKE else ["ripeness", "firmness", "sweetness"],
    deephs_hw=(32, 32) if SMOKE else (64, 64),       # spatial size fed to the models (centre crop / zero pad)
    leaf_hw=(24, 88) if SMOKE else (48, 352),
    normalise="auto",            # "auto": rescale a cube only if it is clearly not reflectance (max > 2)
    cache_gb=2.0 if SMOKE else 16.0,                 # RAM budget for the in-memory cube cache
    workers=0,                                       # 0 keeps data loading deterministic
    split_fracs=(0.6, 0.2, 0.2),                     # specimen-level train / val / test
    # ---- model ----
    d_model=64, n_heads=4, n_layers=2 if SMOKE else 4, spatial_tokens=4,
    # ---- optimisation ----
    batch=8 if SMOKE else 32,
    lr=1e-3, weight_decay=1e-4,
    pretrain_epochs=1 if SMOKE else 100, mask_ratio=0.5,
    epochs=4 if SMOKE else 100, patience=30, min_steps=0 if SMOKE else 300,
    # ---- augmentation (simulates an unseen camera) ----
    aug_fwhm_scale=(1.0, 2.5), aug_shift_nm=3.0, aug_band_drop=0.2,
    # ---- protocols ----
    protocols=["LOCO", "WITHIN", "DAY"],
    run_ablations=True,
    # ---- efficiency study ----
    eff_epochs=1 if SMOKE else 10, eff_repeats=20 if SMOKE else 200,
    energy_min_seconds=2.0 if SMOKE else 30.0, throughput_batch=16,
    country_iso="IND",           # used by CodeCarbon for the grid carbon intensity only
)

# ---- data sources (see Section 3 for verification status) ----
DEEPHS_BASE_URL = "https://cogsys.cs.uni-tuebingen.de/webprojects/DeepHS-Fruit-2023-Datasets/"
DEEPHS_FILES = {  # sizes as listed on the download index
    "annotations": "annotations-upd-2024-01-09.zip",   # 162 KB
    "mango": "Mango.zip",       # 2.7 GB
    "kaki": "Kaki.zip",         # 2.2 GB
    "papaya": "Papaya.zip",     # 22 GB
    "kiwi": "Kiwi.zip",         # 44 GB
    "avocado": "Avocado.zip",   # 72 GB
}
KAGGLE_COMPETITION = "HyperLeaf2024"

# Camera wavelength grids: end points and band counts from cogsys-tuebingen/hsi_benchmark
# (camera_definitions.py). The real per-band centres are read from each ENVI header when present;
# this table is the fallback and the basis of the synthetic data.
CAMERA_GRIDS = {
    "SPECIM_FX10": (397.66, 1003.81, 224),
    "CORNING_HSI": (408.034, 901.256, 249),
    "INNOSPEC_REDEYE": (919.678, 1727.04, 252),
}
# Folder tokens reported in a third-party fork README (NOT VERIFIED against the official data).
CAMERA_ALIASES = {"VIS": "SPECIM_FX10", "VIS_COR": "CORNING_HSI", "NIR": "INNOSPEC_REDEYE",
                  "FX10": "SPECIM_FX10", "CORNING": "CORNING_HSI", "REDEYE": "INNOSPEC_REDEYE"}

# Literature values. Quoted, never recomputed here, and NOT protocol-matched to this benchmark
# (published DeepHS splits are not specimen-disjoint and use very small test sets).
REFERENCE_NUMBERS = pd.DataFrame(
    [
        ("DeepHS Fruit", "Fruit-HSNet overall accuracy (%)", 70.73, "arXiv:2608.01202"),
        ("DeepHS Fruit", "DeepHS-Net overall accuracy (%)", 58.28, "arXiv:2608.01202, Table 1"),
        ("DeepHS Fruit", "DeepHS-Net + HyveConv overall accuracy (%)", 57.57, "arXiv:2608.01202, Table 1"),
        ("DeepHS Fruit", "SpectralFormer overall accuracy (%)", 41.71, "arXiv:2608.01202, Table 1"),
        ("HyperLeaf2024", "ConvNeXt-S 2D-pretrained cultivar accuracy", 0.893, "Laprade et al., CVPRW 2024"),
        ("HyperLeaf2024", "ConvNeXt-S 1D mean standardised MSE", 0.383, "Laprade et al., CVPRW 2024"),
    ],
    columns=["dataset", "quantity", "value", "source"],
)

print(f"RUN_MODE={RUN_MODE} | data={DATA_SOURCE} | seeds={SEEDS} | results -> {OUT_DIR}")

: 

## 2. Environment, determinism and hardware report

Install once (GPU users: install the PyTorch build that matches your CUDA version first):

```bash
pip install torch numpy pandas scipy scikit-learn scikit-posthocs spectral tifffile \
            lightgbm codecarbon fvcore matplotlib psutil kaggle
```

In [ ]:
import psutil
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


def set_seed(seed: int) -> None:
    """Seed Python, NumPy and PyTorch and request deterministic kernels."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.use_deterministic_algorithms(True, warn_only=True)


def cpu_name() -> str:
    try:
        for line in open("/proc/cpuinfo"):
            if line.lower().startswith("model name"):
                return line.split(":", 1)[1].strip()
    except OSError:
        pass
    return platform.processor() or "unknown"


def hardware_report() -> dict:
    """Hardware and software description to quote in the paper's experimental setup."""
    rep = dict(
        python=platform.python_version(), torch=torch.__version__, numpy=np.__version__,
        os=platform.platform(), cpu=cpu_name(),
        physical_cores=psutil.cpu_count(logical=False), logical_cores=psutil.cpu_count(logical=True),
        torch_threads=torch.get_num_threads(),
        ram_gb=round(psutil.virtual_memory().total / 1e9, 1),
        cuda_available=torch.cuda.is_available(),
    )
    if torch.cuda.is_available():
        prop = torch.cuda.get_device_properties(0)
        rep.update(gpu=prop.name, vram_gb=round(prop.total_memory / 1e9, 1), cuda=torch.version.cuda)
    (OUT_DIR / "hardware.json").write_text(json.dumps(rep, indent=2))
    return rep


# The transformer inference fast path is switched off so that FP32 and INT8 models run the same code
# path (the fast path cannot execute dynamically quantised layers).
if hasattr(torch.backends, "mha"):
    torch.backends.mha.set_fastpath_enabled(False)
warnings.filterwarnings("ignore", message=".*does not have a deterministic implementation.*")
warnings.filterwarnings("ignore", message=".*enable_nested_tensor.*")
warnings.filterwarnings("ignore", category=FutureWarning, module=r"torch\.jit.*")
set_seed(0)
HARDWARE = hardware_report()
pd.Series(HARDWARE)

: 

## 3. Data acquisition

| Dataset | What is verified | What is **NOT VERIFIED** |
|---|---|---|
| **DeepHS Fruit v2** (Tübingen) | Per-fruit zip files and sizes on the download index; three cameras and their wavelength ranges / band counts; three-class ripeness, firmness and sweetness labels; no licence stated | Field names inside the annotation zip; raw class strings; directory tree and ENVI file extensions; per-band FWHM |
| **HyperLeaf2024** (Kaggle) | 2,410 wheat-leaf cubes, 48×352×204, 16-bit TIFF, reflectance clipped to 0–1, non-leaf pixels zero; targets `GrainWeight`, `Gsw`, `PhiPS2`, `Fertilizer`; four cultivars; plot-level 24/4/8 split in the paper | CSV file and column names for image ID, plot and cultivar; the exact 204 band centres; whether labelled test plots are public; data licence (read the Kaggle Rules tab) |

Because DeepHS Fruit states no licence, **publish only code and split indices, never the cubes**.

The synthetic generator below writes ENVI (`.hdr` + `.bin`) fruit cubes and 16-bit TIFF leaf cubes with the
same shapes, wavelength grids and labelling pattern (a fruit is labelled only on its last imaging day), so
the whole pipeline can be verified before the large download.

In [ ]:
def download(url: str, dst: Path, chunk: int = 1 << 22) -> Path:
    """Resumable HTTP download. Skips the file when it is already complete."""
    head = urllib.request.Request(url, method="HEAD")
    total = int(urllib.request.urlopen(head).headers.get("Content-Length", 0))
    have = dst.stat().st_size if dst.exists() else 0
    if total and have >= total:
        return dst
    req = urllib.request.Request(url, headers={"Range": f"bytes={have}-"})
    with urllib.request.urlopen(req) as resp, open(dst, "ab") as fh:
        while block := resp.read(chunk):
            fh.write(block)
    return dst


def fetch_deephs(fruits: list[str]) -> None:
    """Download and unpack the annotation zip and the requested fruit zips."""
    for key in ["annotations", *fruits]:
        archive = DEEPHS_DIR / DEEPHS_FILES[key]
        print("fetching", archive.name)
        download(DEEPHS_BASE_URL + archive.name, archive)
        target = DEEPHS_DIR / archive.stem
        if not target.exists():
            with zipfile.ZipFile(archive) as zf:
                zf.extractall(target)


def fetch_hyperleaf() -> None:
    """Needs ~/.kaggle/kaggle.json and the competition rules accepted on the Kaggle website."""
    subprocess.run(["kaggle", "competitions", "download", "-c", KAGGLE_COMPETITION, "-p", str(HLEAF_DIR)],
                   check=True)
    for archive in HLEAF_DIR.glob("*.zip"):
        with zipfile.ZipFile(archive) as zf:
            zf.extractall(HLEAF_DIR)


def camera_grid(camera: str) -> np.ndarray:
    lo, hi, n = CAMERA_GRIDS[camera]
    return np.linspace(lo, hi, n, dtype=np.float32)


# Cameras available per fruit, as reported for DeepHS Fruit v2 (used for the synthetic data only).
SYNTHETIC_CAMERAS = {
    "avocado": ["SPECIM_FX10", "CORNING_HSI", "INNOSPEC_REDEYE"],
    "kiwi": ["SPECIM_FX10", "INNOSPEC_REDEYE"],
    "mango": ["SPECIM_FX10", "CORNING_HSI"],
    "kaki": ["SPECIM_FX10", "CORNING_HSI"],
    "papaya": ["SPECIM_FX10", "CORNING_HSI"],
}
CLASS_NAMES = {
    "ripeness": ["unripe", "ripe", "overripe"],
    "firmness": ["too_firm", "perfect", "too_soft"],
    "sweetness": ["not_sweet", "sweet", "overly_sweet"],
}


def make_synthetic_deephs(n_fruits: int = 18, n_days: int = 3, size: int = 32, seed: int = 0) -> None:
    """Write synthetic ENVI cubes plus an annotation CSV that mimic DeepHS Fruit.

    The label changes the depth of a visible absorption (680 nm) and of a short-wave infrared
    absorption (1450 nm), so every camera carries signal. Each camera has its own gain and offset.
    """
    import spectral.io.envi as envi

    rng = np.random.default_rng(seed)
    cam_gain = {"SPECIM_FX10": (1.00, 0.00), "CORNING_HSI": (0.85, 0.03), "INNOSPEC_REDEYE": (1.10, 0.01)}
    yy, xx = np.mgrid[:size, :size]
    mask = ((yy - size / 2) ** 2 + (xx - size / 2) ** 2) < (size / 2.3) ** 2
    rows = []
    for fruit in CFG["fruits"]:
        for fid in range(n_fruits):
            label = fid % 3                                  # balanced classes
            last_day = int(rng.integers(1, n_days))          # destructive test ends the series
            for day in range(last_day + 1):
                ripeness = label / 2.0 + 0.1 * day / max(last_day, 1)
                for cam in SYNTHETIC_CAMERAS[fruit]:
                    wl = camera_grid(cam)
                    gain, offset = cam_gain[cam]
                    spec = (0.30 + 0.20 * np.tanh((wl - 720) / 40)
                            - 0.25 * (1 - ripeness) * np.exp(-((wl - 680) / 25) ** 2)
                            - 0.20 * ripeness * np.exp(-((wl - 1450) / 60) ** 2)
                            + 0.02 * np.sin(wl / 90 + fid))
                    spec = gain * spec + offset
                    for side in ("front", "back"):
                        cube = mask[..., None] * spec[None, None, :]
                        cube = cube * (1 + 0.04 * rng.standard_normal((size, size, 1)))
                        cube = cube + mask[..., None] * 0.015 * rng.standard_normal(cube.shape)
                        name = f"{fruit}_{cam}_day{day:02d}_{fid:03d}_{side}"
                        hdr = SYN_DIR / "deephs" / fruit / cam / f"{name}.hdr"
                        hdr.parent.mkdir(parents=True, exist_ok=True)
                        envi.save_image(str(hdr), cube.astype(np.float32), ext=".bin", force=True,
                                        metadata={"wavelength": [float(w) for w in wl],
                                                  "wavelength units": "nm"})
                        labelled = day == last_day
                        row = dict(path=str(hdr), fruit=fruit, camera=cam, fruit_id=f"{fruit}_{fid:03d}",
                                   day=day, side=side)
                        for task in CLASS_NAMES:
                            has = labelled and not (task == "sweetness" and fruit == "avocado")
                            row[task] = CLASS_NAMES[task][label] if has else None
                        rows.append(row)
    pd.DataFrame(rows).to_csv(SYN_DIR / "deephs" / "annotations.csv", index=False)


CULTIVARS = ["Heerup", "Kvium", "Rembrandt", "Sheriff"]
LEAF_TARGETS = ["GrainWeight", "Gsw", "PhiPS2", "Fertilizer"]


def make_synthetic_hyperleaf(n_plots: int = 36, leaves_per_plot: int = 4, seed: int = 0) -> None:
    """Write synthetic 16-bit TIFF leaf cubes (bands first) plus a label CSV.

    The CSV column names for image ID, plot and cultivar are placeholders for the synthetic data;
    the real names are discovered automatically in Section 4.
    """
    import tifffile

    rng = np.random.default_rng(seed)
    h, w = CFG["leaf_hw"]
    wl = np.linspace(400, 1000, 204)
    img_dir = SYN_DIR / "hyperleaf" / "images"
    img_dir.mkdir(parents=True, exist_ok=True)
    rows = []
    for plot in range(n_plots):
        cultivar, fert = plot % 4, (plot // 4) % 3
        plot_effect = rng.normal(0, 0.2)
        for leaf in range(leaves_per_plot):
            red_edge = 712 + 4 * cultivar
            spec = (0.05 + 0.45 / (1 + np.exp(-(wl - red_edge) / 12))
                    + (0.08 - 0.02 * fert) * np.exp(-((wl - 550) / 30) ** 2))
            cube = spec[:, None, None] * (1 + 0.05 * rng.standard_normal((204, h, w)))
            cube = np.clip(cube, 0, 1)
            cube[:, :3, :] = 0
            cube[:, -3:, :] = 0                                   # non-leaf pixels are zero
            image_id = f"plot{plot:02d}_leaf{leaf}"
            tifffile.imwrite(img_dir / f"{image_id}.tiff", (cube * 65535).astype(np.uint16))
            row = dict(ImageId=image_id, PlotId=plot,
                       GrainWeight=5 + fert + plot_effect + rng.normal(0, 0.3),
                       Gsw=0.3 + 0.03 * fert + rng.normal(0, 0.08),
                       PhiPS2=0.60 + 0.03 * fert + rng.normal(0, 0.02),
                       Fertilizer=float(fert))
            row.update({c: int(i == cultivar) for i, c in enumerate(CULTIVARS)})
            rows.append(row)
    pd.DataFrame(rows).to_csv(SYN_DIR / "hyperleaf" / "train.csv", index=False)


if USE_SYNTHETIC:
    if not (SYN_DIR / "deephs" / "annotations.csv").exists():
        make_synthetic_deephs(size=CFG["deephs_hw"][0])
    if not (SYN_DIR / "hyperleaf" / "train.csv").exists():
        make_synthetic_hyperleaf()
    print("synthetic data ready in", SYN_DIR)
else:
    # Comment these out once the data are on disk.
    fetch_deephs(CFG["fruits"])
    fetch_hyperleaf()

## 4. Dataset indexing and schema discovery

Every recording becomes one row of a canonical index:
`path, fruit, camera, fruit_id, day, side, ripeness, firmness, sweetness`.

**Real DeepHS Fruit — one manual check is required.** The annotation field names could not be verified,
so nothing is hard-coded. `inspect_annotations()` prints what is actually in the files, and
`FIELD_CANDIDATES` lists the raw keys tried for each canonical field (flattened with dots, e.g.
`label.ripeness_state`). The candidates are educated guesses: confirm the printed mapping, and edit the
lists if a field is not found. `fruit_id` is the leakage unit, so it gets extra sanity checks.

In [ ]:
def inspect_annotations(root: Path, n: int = 2) -> None:
    """Print the structure of every JSON / CSV annotation file under `root`."""
    for f in sorted(root.rglob("*")):
        if f.suffix.lower() == ".json":
            obj = json.loads(f.read_text())
            recs = obj.get("records", obj) if isinstance(obj, dict) else obj
            sample = recs[:n] if isinstance(recs, list) else dict(list(recs.items())[:n])
            print(f"== {f.relative_to(root)} ({len(recs)} entries)")
            print(json.dumps(sample, indent=1, default=str)[:1500])
        elif f.suffix.lower() == ".csv":
            print(f"== {f.relative_to(root)}")
            print(pd.read_csv(f, nrows=n).to_string())


# canonical field -> raw keys to try, in order (NOT VERIFIED; edit after inspect_annotations()).
FIELD_CANDIDATES = dict(
    path=["files.header_file", "header_file", "path", "file", "filename"],
    fruit=["fruit", "fruit_type"],
    camera=["camera_type", "camera"],
    fruit_id=["fruit_id", "object_id", "id_fruit", "fruit_number"],
    day=["day", "measurement_day", "series_day"],
    side=["side"],
    ripeness=["label.ripeness_state", "ripeness_state", "ripeness"],
    firmness=["label.firmness_state", "firmness_state", "firmness"],
    sweetness=["label.sweetness_state", "sweetness_state", "sweetness", "label.sugar_state"],
)
# Used only when no fruit-ID key exists: regular expression applied to the file stem. NOT VERIFIED.
FRUIT_ID_REGEX = r"(\d+)_(?:front|back)"


def _load_annotation_records(root: Path) -> pd.DataFrame:
    frames = []
    for f in sorted(root.rglob("*.json")):
        obj = json.loads(f.read_text())
        recs = obj.get("records", obj) if isinstance(obj, dict) else obj
        if isinstance(recs, list) and recs and isinstance(recs[0], dict):
            frames.append(pd.json_normalize(recs))
    for f in sorted(root.rglob("*.csv")):
        frames.append(pd.read_csv(f))
    if not frames:
        raise FileNotFoundError(f"No annotation files under {root}. Run fetch_deephs() first.")
    return pd.concat(frames, ignore_index=True)


def build_deephs_index_real(root: Path) -> pd.DataFrame:
    raw = _load_annotation_records(root)
    chosen, out = {}, pd.DataFrame(index=raw.index)
    for field, candidates in FIELD_CANDIDATES.items():
        key = next((c for c in candidates if c in raw.columns), None)
        chosen[field] = key
        if key is not None:
            out[field] = raw[key]
    print("Schema mapping (confirm against inspect_annotations):")
    for k, v in chosen.items():
        print(f"  {k:10s} <- {v}")
    required = ["path", "fruit", "camera"]
    missing = [k for k in required if chosen[k] is None]
    if missing:
        raise KeyError(f"Could not map {missing}. Available raw columns: {sorted(raw.columns)}. "
                       "Edit FIELD_CANDIDATES.")
    # Resolve header files on disk: match the annotated relative path, fall back to a unique file name.
    by_name: dict[str, list[Path]] = {}
    for hdr in root.rglob("*.hdr"):
        by_name.setdefault(hdr.name, []).append(hdr)

    def resolve(value) -> str:
        rel = Path(str(value).replace("\\", "/"))
        candidates = by_name.get(rel.name, [])
        exact = [c for c in candidates if c.as_posix().endswith(rel.as_posix())]
        if len(exact) == 1:
            return str(exact[0])
        return str(candidates[0]) if len(candidates) == 1 else ""

    out["path"] = out["path"].map(resolve)
    n_missing = int((out["path"] == "").sum())
    if n_missing:
        warnings.warn(f"{n_missing} annotated recordings were not found on disk (fruit not downloaded, "
                      "or ambiguous file names) and are skipped.")
    out = out[out["path"] != ""].drop_duplicates("path").reset_index(drop=True)
    out["fruit"] = out["fruit"].astype(str).str.lower()
    out["camera"] = out["camera"].astype(str).str.upper().replace(CAMERA_ALIASES)
    if "fruit_id" not in out:
        import re
        ids = out["path"].map(lambda s: (re.search(FRUIT_ID_REGEX, Path(s).stem) or [None, None])[1])
        if ids.isna().any():
            raise KeyError("No fruit-ID key and FRUIT_ID_REGEX did not match every file name. "
                           "Edit FIELD_CANDIDATES['fruit_id'] or FRUIT_ID_REGEX.")
        out["fruit_id"] = ids
    out["fruit_id"] = out["fruit"] + "_" + out["fruit_id"].astype(str)
    for col in ("day", "side", *CLASS_NAMES):
        if col not in out:
            out[col] = None
    return out


def validate_deephs_index(df: pd.DataFrame) -> None:
    """Sanity checks on the leakage unit. A wrong fruit ID would silently break every split."""
    dup = df.groupby(["fruit_id", "camera", "day", "side"], dropna=False).size().max()
    assert dup == 1, ("fruit_id is not unique per (camera, day, side): the ID mapping is wrong "
                      f"(max duplicates = {dup}).")
    for task in CLASS_NAMES:
        per_fruit = df.dropna(subset=[task]).groupby("fruit_id")[task].nunique()
        assert (per_fruit <= 1).all(), f"A fruit carries more than one {task} label: check fruit_id."
        n_unique = df[task].dropna().nunique()
        if n_unique > 10:
            warnings.warn(f"'{task}' has {n_unique} distinct values and looks continuous; "
                          "it will be skipped until it is binned into classes.")


def build_deephs_index() -> pd.DataFrame:
    if USE_SYNTHETIC:
        df = pd.read_csv(SYN_DIR / "deephs" / "annotations.csv")
    else:
        df = build_deephs_index_real(DEEPHS_DIR)
    for task in CLASS_NAMES:                       # normalise missing labels to None
        df[task] = df[task].astype(object).where(df[task].notna(), None)
    df = df[df["fruit"].isin(CFG["fruits"])].reset_index(drop=True)
    validate_deephs_index(df)
    return df


# ---- HyperLeaf2024 ----
HLEAF_SCHEMA = dict(id_col=None, plot_col=None, cultivar_col=None)   # None = auto-detect


def build_hyperleaf_index() -> pd.DataFrame:
    base = SYN_DIR / "hyperleaf" if USE_SYNTHETIC else HLEAF_DIR
    tables = [(f, pd.read_csv(f)) for f in sorted(base.rglob("*.csv"))]
    tables = [(f, t) for f, t in tables if set(LEAF_TARGETS) <= set(t.columns)]
    if not tables:
        raise FileNotFoundError(f"No CSV with the target columns {LEAF_TARGETS} under {base}.")
    csv_path, df = max(tables, key=lambda ft: len(ft[1]))
    lower = {c.lower(): c for c in df.columns}

    id_col = HLEAF_SCHEMA["id_col"] or next(
        (lower[k] for k in ("imageid", "image_id", "id", "image", "filename", "file") if k in lower), None)
    plot_col = HLEAF_SCHEMA["plot_col"] or next((c for c in df.columns if "plot" in c.lower()), None)
    if id_col is None or plot_col is None:
        raise KeyError(f"Could not detect the image-ID / plot columns in {csv_path.name} "
                       f"(columns: {list(df.columns)}). Set HLEAF_SCHEMA. Without a plot identifier "
                       "the plot-level protocol cannot be reproduced.")
    if set(CULTIVARS) <= set(df.columns):                       # one-hot cultivar columns
        cultivar = df[CULTIVARS].to_numpy().argmax(1)
        cultivar = pd.Series([CULTIVARS[i] for i in cultivar], index=df.index)
    else:
        col = HLEAF_SCHEMA["cultivar_col"] or next(
            (c for c in df.columns if "cultivar" in c.lower() or "variety" in c.lower()), None)
        cultivar = df[col].astype(str) if col else pd.Series([None] * len(df), index=df.index)

    images = {p.stem: p for p in base.rglob("*") if p.suffix.lower() in (".tif", ".tiff", ".npy")}
    out = pd.DataFrame({
        "path": df[id_col].map(lambda s: str(images.get(Path(str(s)).stem, ""))),
        "plot": df[plot_col], "cultivar": cultivar, "camera": "SPECIM_IQ",
    })
    out[LEAF_TARGETS] = df[LEAF_TARGETS].astype(float)
    out = out[out["path"] != ""].dropna(subset=LEAF_TARGETS).reset_index(drop=True)
    print(f"HyperLeaf index from {csv_path.name}: id='{id_col}', plot='{plot_col}', "
          f"{len(out)} labelled cubes, {out['plot'].nunique()} plots")
    return out


if not USE_SYNTHETIC:
    inspect_annotations(DEEPHS_DIR)

DEEPHS = build_deephs_index()
HLEAF = build_hyperleaf_index()
print("\nDeepHS recordings per fruit and camera (labelled / all):")
print(DEEPHS.groupby(["fruit", "camera"]).agg(labelled=("ripeness", "count"), recordings=("path", "size")))

## 5. Leakage-free splits

The unit of independence is the **specimen** (one physical fruit, or one field plot). Front and back
views, repeated days and different cameras of the same fruit must never straddle a split.

For every fruit, task and seed the specimens are first divided into train / validation / test groups
(stratified by label). The protocols then differ only in which *cameras* feed the training set:

| Protocol | Training data | Test data |
|---|---|---|
| `LOCO` | train specimens, **all cameras except the held-out one** | test specimens, held-out camera |
| `WITHIN` | train specimens, held-out camera only | *the same* test recordings |
| `DAY` | earlier measurement days, all cameras | latest days, disjoint specimens |

`LOCO` and `WITHIN` are scored on identical test recordings, so the per-camera drop
(`WITHIN − LOCO`) is a paired quantity.

Each fold is also tagged with a **tier**, from the fraction of the held-out camera's wavelength range that
the training cameras also measure (`coverage`): `overlap` when it is at least one half (FX10 ↔ Corning),
otherwise `extrapolation`. The RedEye (920–1727 nm) shares only about 84 nm with the FX10 (398–1004 nm)
and nothing with the Corning (408–901 nm), so folds that cross between the RedEye and a visible camera
are extrapolation and near-chance accuracy is the expected outcome there. Report the two tiers separately.

In [ ]:
def assert_disjoint(a: pd.DataFrame, b: pd.DataFrame, key: str, what: str = "") -> None:
    shared = set(a[key]) & set(b[key])
    assert not shared, f"LEAKAGE {what}: {len(shared)} shared '{key}' values, e.g. {sorted(shared)[:3]}"


def partition_specimens(d: pd.DataFrame, task: str, seed: int) -> dict[str, set]:
    """Stratified specimen-level train / val / test partition for one fruit and task."""
    rng = np.random.default_rng(seed)
    label_of = d.groupby("fruit_id")[task].first()
    parts = {"train": set(), "val": set(), "test": set()}
    f_train, f_val, _ = CFG["split_fracs"]
    for _, ids in label_of.groupby(label_of):
        ids = np.array(sorted(ids.index))
        rng.shuffle(ids)
        n = len(ids)
        n_test = max(1, round(n * (1 - f_train - f_val))) if n >= 3 else 0
        n_val = max(1, round(n * f_val)) if n >= 3 else 0
        parts["test"] |= set(ids[:n_test])
        parts["val"] |= set(ids[n_test:n_test + n_val])
        parts["train"] |= set(ids[n_test + n_val:])
    return parts


def spectral_coverage(held_out: str, train_cameras: list[str]) -> float:
    """Fraction of the held-out camera's wavelength range that the training cameras also measure."""
    lo, hi, _ = CAMERA_GRIDS[held_out]
    spans = sorted((max(lo, CAMERA_GRIDS[c][0]), min(hi, CAMERA_GRIDS[c][1])) for c in train_cameras)
    covered, cursor = 0.0, lo
    for a, b in spans:
        if b > max(a, cursor):
            covered += b - max(a, cursor)
            cursor = b
    return covered / (hi - lo)


def spectral_tier(held_out: str, train_cameras: list[str]) -> str:
    """'overlap' when at least half of the held-out camera's range is seen in training."""
    return "overlap" if spectral_coverage(held_out, train_cameras) >= 0.5 else "extrapolation"


def loco_folds(df: pd.DataFrame, fruit: str, task: str, seed: int) -> list[dict]:
    """Leave-one-camera-out folds with a paired within-camera reference."""
    d = df[(df["fruit"] == fruit) & df[task].notna()]
    cameras = sorted(d["camera"].unique())
    if len(cameras) < 2:
        return []
    parts = partition_specimens(d, task, seed)
    in_part = {k: d["fruit_id"].isin(v) for k, v in parts.items()}
    all_fruit = df[df["fruit"] == fruit]
    folds = []
    for held in cameras:
        is_held = d["camera"] == held
        fold = dict(
            held_out=held, tier=spectral_tier(held, [c for c in cameras if c != held]),
            coverage=spectral_coverage(held, [c for c in cameras if c != held]),
            test=d[is_held & in_part["test"]],
            loco_train=d[~is_held & in_part["train"]], loco_val=d[~is_held & in_part["val"]],
            within_train=d[is_held & in_part["train"]], within_val=d[is_held & in_part["val"]],
        )
        # Pretraining pools: spectra only (labels unused), never test/val specimens,
        # and for LOCO never the held-out camera (that would be a transductive leak).
        free = ~all_fruit["fruit_id"].isin(parts["test"] | parts["val"])
        fold["loco_pretrain"] = all_fruit[free & (all_fruit["camera"] != held)]
        fold["within_pretrain"] = all_fruit[free & (all_fruit["camera"] == held)]
        if min(len(fold["test"]), len(fold["loco_train"]), len(fold["within_train"])) == 0:
            continue
        # ---- leakage assertions ----
        for name in ("loco_train", "loco_val", "within_train", "within_val",
                     "loco_pretrain", "within_pretrain"):
            assert_disjoint(fold[name], fold["test"], "fruit_id", f"{name} vs test")
        assert_disjoint(fold["loco_train"], fold["loco_val"], "fruit_id", "train vs val")
        assert held not in set(fold["loco_train"]["camera"]) | set(fold["loco_val"]["camera"])
        assert held not in set(fold["loco_pretrain"]["camera"])
        assert set(fold["test"]["camera"]) == {held}
        folds.append(fold)
    return folds


def day_disjoint_fold(df: pd.DataFrame, fruit: str, task: str, seed: int) -> dict | None:
    """Train on earlier measurement days, test on the latest ones; specimen- and day-disjoint."""
    d = df[(df["fruit"] == fruit) & df[task].notna() & df["day"].notna()]
    days = sorted(d["day"].unique())
    if len(days) < 2:
        return None
    late = set(days[-max(1, len(days) // 3):])
    test = d[d["day"].isin(late)]
    rest = d[~d["day"].isin(late) & ~d["fruit_id"].isin(test["fruit_id"])]
    ids = np.array(sorted(rest["fruit_id"].unique()))
    np.random.default_rng(seed).shuffle(ids)
    val_ids = set(ids[:max(1, round(len(ids) * CFG["split_fracs"][1]))]) if len(ids) >= 3 else set()
    fold = dict(test=test, train=rest[~rest["fruit_id"].isin(val_ids)], val=rest[rest["fruit_id"].isin(val_ids)])
    if min(len(fold["test"]), len(fold["train"])) == 0:
        return None
    assert_disjoint(fold["train"], fold["test"], "fruit_id", "day: train vs test")
    assert_disjoint(fold["train"], fold["test"], "day", "day: train vs test")
    assert_disjoint(fold["val"], fold["test"], "fruit_id", "day: val vs test")
    assert_disjoint(fold["train"], fold["val"], "fruit_id", "day: train vs val")
    return fold


def plot_level_split(df: pd.DataFrame, seed: int, ratio: tuple = (24, 4, 8)) -> dict:
    """Plot-level split in the 24/4/8 proportion of the HyperLeaf2024 paper.

    The authors' exact plot assignment is not public in a verified form, so plots are drawn at random
    per seed. State this deviation in the paper.
    """
    plots = np.array(sorted(df["plot"].unique()))
    np.random.default_rng(seed).shuffle(plots)
    n_train = round(len(plots) * ratio[0] / sum(ratio))
    n_val = max(1, round(len(plots) * ratio[1] / sum(ratio)))
    groups = dict(train=set(plots[:n_train]), val=set(plots[n_train:n_train + n_val]),
                  test=set(plots[n_train + n_val:]))
    split = {k: df[df["plot"].isin(v)] for k, v in groups.items()}
    for a, b in (("train", "val"), ("train", "test"), ("val", "test")):
        assert_disjoint(split[a], split[b], "plot", f"{a} vs {b}")
    return split


# Quick audit of the folds that will be run.
_audit = []
for _fruit in CFG["fruits"]:
    for _task in CFG["tasks"]:
        for _f in loco_folds(DEEPHS, _fruit, _task, seed=0):
            _audit.append(dict(fruit=_fruit, task=_task, held_out=_f["held_out"], tier=_f["tier"],
                               coverage=round(_f["coverage"], 2),
                               n_loco_train=len(_f["loco_train"]), n_within_train=len(_f["within_train"]),
                               n_test=len(_f["test"]), n_pretrain=len(_f["loco_pretrain"])))
FOLD_AUDIT = pd.DataFrame(_audit)
FOLD_AUDIT

## 6. Cube loading, preprocessing and spectral resampling augmentation

* ENVI cubes are read with Spectral Python; band centres come from the header, and the full width at half
  maximum (FWHM) comes from the header when present. **When the header has no FWHM, the band spacing is
  used as a proxy** — say so in the paper.
* HyperLeaf TIFFs carry no verified wavelength list, so `np.linspace(400, 1000, 204)` is used as an
  approximation of the Specim IQ grid — also to be stated.
* `resample_augment` simulates an unseen camera: it broadens the spectral response, shifts the band
  centres, changes the sampling density and drops bands. The wavelengths returned always describe the
  augmented cube, so the model is never told a wrong wavelength.

In [ ]:
from scipy.ndimage import gaussian_filter1d

_CUBE_CACHE: dict = {}
_CACHE_BYTES = 0


def _fit_hw(cube: np.ndarray, hw: tuple) -> np.ndarray:
    """Centre-crop or zero-pad the two spatial axes of an (H, W, B) cube to `hw`."""
    out = np.zeros((*hw, cube.shape[2]), dtype=np.float32)
    h, w = min(hw[0], cube.shape[0]), min(hw[1], cube.shape[1])
    y0, x0 = (cube.shape[0] - h) // 2, (cube.shape[1] - w) // 2
    yo, xo = (hw[0] - h) // 2, (hw[1] - w) // 2
    out[yo:yo + h, xo:xo + w] = cube[y0:y0 + h, x0:x0 + w]
    return out


def read_cube(path: str, camera: str | None = None):
    """Return (cube[H, W, B] float32, centre wavelengths [B] nm, FWHM [B] nm)."""
    if path.endswith(".hdr"):
        import spectral.io.envi as envi
        img = envi.open(path)
        cube = np.asarray(img.load(), dtype=np.float32)
        centres = img.bands.centers
        if centres is None:
            if camera in CAMERA_GRIDS and CAMERA_GRIDS[camera][2] == cube.shape[2]:
                centres = camera_grid(camera)
            else:
                raise ValueError(f"{path}: no wavelengths in the header and no matching camera grid.")
        wl = np.asarray(centres, dtype=np.float32)
        if float(wl.max()) < 20:                      # header in micrometres
            wl = wl * 1000.0
        widths = img.bands.bandwidths
        fwhm = np.asarray(widths, dtype=np.float32) if widths is not None else np.gradient(wl)
    else:
        if path.endswith(".npy"):
            cube = np.load(path).astype(np.float32)
        else:
            import tifffile
            raw = tifffile.imread(path)
            cube = raw.astype(np.float32) / (65535.0 if raw.dtype == np.uint16 else 1.0)
        band_axis = int(np.argmax([s == 204 for s in cube.shape])) if 204 in cube.shape else 0
        cube = np.moveaxis(cube, band_axis, -1)
        # APPROXIMATION: exact Specim IQ band centres are not verified.
        wl = np.linspace(400, 1000, cube.shape[2], dtype=np.float32)
        fwhm = np.gradient(wl)
    if CFG["normalise"] == "auto" and float(cube.max()) > 2.0:   # raw counts, not reflectance
        cube = cube / max(float(np.percentile(cube[cube > 0], 99.9)), 1e-6)
    return cube, wl, fwhm.astype(np.float32)


def load_cube(path: str, camera: str | None, hw: tuple):
    """Cached, spatially standardised cube."""
    global _CACHE_BYTES
    key = (path, hw)
    if key not in _CUBE_CACHE:
        cube, wl, fwhm = read_cube(path, camera)
        item = (_fit_hw(cube, hw), wl, fwhm)
        if _CACHE_BYTES + item[0].nbytes > CFG["cache_gb"] * 1e9:
            return item
        _CUBE_CACHE[key] = item
        _CACHE_BYTES += item[0].nbytes
    return _CUBE_CACHE[key]


def resample_augment(cube, wl, fwhm, rng):
    """Simulate a different spectral sensor. Returns a new (cube, wl, fwhm)."""
    spacing = float(np.median(np.diff(wl)))
    # 1) broaden the spectral response so that the effective FWHM becomes k * FWHM
    k = rng.uniform(*CFG["aug_fwhm_scale"])
    sigma_bands = math.sqrt(max(k * k - 1.0, 0.0)) * float(np.median(fwhm)) / 2.355 / spacing
    if sigma_bands > 0.05:
        cube = gaussian_filter1d(cube, sigma_bands, axis=-1, mode="nearest")
    # 2) shift the band centres by a sub-band amount (linear interpolation between neighbours)
    shift = rng.uniform(-CFG["aug_shift_nm"], CFG["aug_shift_nm"])
    t = float(np.clip(shift / spacing, -0.9, 0.9))
    neighbour = np.roll(cube, -1 if t >= 0 else 1, axis=-1)
    edge = -1 if t >= 0 else 0
    neighbour[..., edge] = cube[..., edge]
    cube = (1 - abs(t)) * cube + abs(t) * neighbour
    wl = wl + t * spacing
    # 3) change the sampling density and drop random bands
    stride = int(rng.choice([1, 1, 2, 3]))
    keep = np.arange(int(rng.integers(stride)), len(wl), stride)
    if len(keep) > 32:
        keep = keep[rng.random(len(keep)) > CFG["aug_band_drop"]]
    return (np.ascontiguousarray(cube[..., keep], dtype=np.float32), wl[keep].astype(np.float32),
            (fwhm[keep] * k).astype(np.float32))


class CubeDataset(Dataset):
    """Hyperspectral cubes with their wavelengths.

    y_cols: None (unlabelled), a column name holding integer class indices, or a list of float columns.
    subset: optional (low_nm, high_nm) window applied at load time (band-subset shift).
    """

    def __init__(self, df, hw, y_cols=None, augment=False, seed=0, subset=None):
        self.df, self.hw, self.y_cols = df.reset_index(drop=True), hw, y_cols
        self.augment, self.subset = augment, subset
        self.rng = np.random.default_rng(seed)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, i):
        row = self.df.iloc[i]
        cube, wl, fwhm = load_cube(row["path"], row.get("camera"), self.hw)
        if self.subset is not None:
            keep = (wl >= self.subset[0]) & (wl <= self.subset[1])
            cube, wl, fwhm = cube[..., keep], wl[keep], fwhm[keep]
        if self.augment:
            cube, wl, fwhm = resample_augment(cube, wl, fwhm, self.rng)
            if self.rng.random() < 0.5:
                cube = cube[::-1]
            if self.rng.random() < 0.5:
                cube = cube[:, ::-1]
        if self.y_cols is None:
            y = torch.zeros((), dtype=torch.long)
        elif isinstance(self.y_cols, str):
            y = torch.tensor(int(row[self.y_cols]), dtype=torch.long)
        else:
            y = torch.tensor(row[self.y_cols].to_numpy(dtype=np.float32))
        x = torch.from_numpy(np.ascontiguousarray(cube)).permute(2, 0, 1)        # (B, H, W)
        return x, torch.from_numpy(np.ascontiguousarray(wl)), torch.from_numpy(np.ascontiguousarray(fwhm)), y


def collate_variable_bands(batch):
    """Zero-pad the band axis so that one batch can mix cameras. `mask` marks the real bands."""
    n_max = max(x.shape[0] for x, *_ in batch)
    xs, wls, fws, masks, ys = [], [], [], [], []
    for x, wl, fw, y in batch:
        pad = n_max - x.shape[0]
        xs.append(F.pad(x, (0, 0, 0, 0, 0, pad)))
        wls.append(F.pad(wl, (0, pad)))
        fws.append(F.pad(fw, (0, pad)))
        masks.append(torch.arange(n_max) < x.shape[0])
        ys.append(y)
    return torch.stack(xs), torch.stack(wls), torch.stack(fws), torch.stack(masks), torch.stack(ys)


def make_loader(ds, shuffle: bool, seed: int, batch: int | None = None) -> DataLoader:
    batch = batch or CFG["batch"]
    gen = torch.Generator()
    gen.manual_seed(seed)
    return DataLoader(ds, batch_size=batch, shuffle=shuffle, collate_fn=collate_variable_bands,
                      generator=gen, num_workers=CFG["workers"],
                      drop_last=shuffle and len(ds) > batch and len(ds) % batch == 1)


def foreground_stats(x: torch.Tensor):
    """Per-band mean and standard deviation over non-zero (object) pixels. x: (N, B, H, W)."""
    fg = (x.abs().sum(1, keepdim=True) > 0).float()
    n = fg.sum((2, 3)).clamp(min=1.0)
    mean = (x * fg).sum((2, 3)) / n
    var = (((x - mean[..., None, None]) ** 2) * fg).sum((2, 3)) / n
    return mean, (var + 1e-8).sqrt()



def standardise_spectrum(spectrum: torch.Tensor, mask: torch.Tensor):
    """Centre and scale each sample's spectrum over its real bands. Returns (z, centre, scale)."""
    m = mask.float()
    n = m.sum(1, keepdim=True).clamp(min=1.0)
    centre = (spectrum * m).sum(1, keepdim=True) / n
    scale = ((((spectrum - centre) ** 2) * m).sum(1, keepdim=True) / n + 1e-8).sqrt()
    return (spectrum - centre) / scale * m, centre, scale

_x, _wl, _fw, _m, _y = next(iter(make_loader(CubeDataset(DEEPHS.head(4), CFG["deephs_hw"], augment=True), False, 0)))
print("batch:", tuple(_x.shape), "| real bands per sample:", _m.sum(1).tolist())

## 7. Proposed model: wavelength-aware spectral tokeniser (WST)

Each spectral band becomes one token:

`token = MLP(band descriptor) ⊙ (1 + W·w) + w`, where `w = MLP(Fourier features of centre wavelength and FWHM)`

The band descriptor (coarse spatial map, foreground mean, spread and first spectral derivative) is
standardised **per cube**, which removes camera gain, offset and illumination level.

The position of a token is therefore its **physical wavelength**, not its index, so cubes with different
band counts and ranges share one encoder. A small transformer over the band tokens feeds a linear head on the `[CLS]` token and the mean band token.
Self-supervised pretraining masks a fraction of the band tokens and reconstructs their descriptors.

In [ ]:
class WavelengthEncoding(nn.Module):
    """Fourier features of centre wavelength and FWHM (both in nm), projected to d_model."""

    def __init__(self, d_model: int, n_freq: int = 8, lo: float = 350.0, hi: float = 1800.0):
        super().__init__()
        self.lo, self.hi = lo, hi
        self.register_buffer("freqs", (2.0 ** torch.arange(n_freq)) * math.pi)
        self.proj = nn.Sequential(nn.Linear(4 * n_freq, d_model), nn.GELU(), nn.Linear(d_model, d_model))

    def forward(self, wl, fwhm):
        a = ((wl - self.lo) / (self.hi - self.lo)).unsqueeze(-1) * self.freqs
        b = (fwhm / 50.0).unsqueeze(-1) * self.freqs
        return self.proj(torch.cat([a.sin(), a.cos(), b.sin(), b.cos()], dim=-1))


class WST(nn.Module):
    """Wavelength-aware spectral tokeniser + transformer encoder.

    use_wavelength=False replaces the wavelength encoding by a learned band-index embedding
    (the 'no wavelength embedding' ablation).
    """

    def __init__(self, n_out: int, d_model: int = 64, n_heads: int = 4, n_layers: int = 4,
                 spatial: int = 4, use_wavelength: bool = True, max_bands: int = 512):
        super().__init__()
        self.spatial, self.use_wavelength = spatial, use_wavelength
        self.n_feat = spatial * spatial + 3
        # FiLM-style fusion: the wavelength code multiplicatively gates the band descriptor, so the
        # meaning of a value depends on where in the spectrum it was measured. (Adding two embeddings
        # cannot express that interaction before the first attention layer.)
        self.value_embed = nn.Sequential(nn.Linear(self.n_feat, d_model), nn.GELU(), nn.Linear(d_model, d_model))
        self.gate = nn.Linear(d_model, d_model)
        self.wavelength_embed = WavelengthEncoding(d_model)
        self.index_embed = nn.Embedding(max_bands, d_model)
        self.cls_token = nn.Parameter(torch.zeros(1, 1, d_model))
        self.mask_token = nn.Parameter(torch.zeros(1, 1, d_model))
        layer = nn.TransformerEncoderLayer(d_model, n_heads, 4 * d_model, dropout=0.1,
                                           batch_first=True, norm_first=True)
        self.encoder = nn.TransformerEncoder(layer, n_layers)
        self.norm = nn.LayerNorm(d_model)
        self.head = nn.Linear(2 * d_model, n_out)
        self.reconstruct = nn.Linear(d_model, self.n_feat)
        nn.init.trunc_normal_(self.cls_token, std=0.02)
        nn.init.trunc_normal_(self.mask_token, std=0.02)

    def band_features(self, x, wl, mask):
        """(N, B, H, W) -> (N, B, spatial² + 3) band descriptors, standardised per sample.

        Each cube is centred and scaled by the mean and standard deviation of its own foreground
        spectrum, which removes camera gain/offset and illumination level and leaves spectral shape.
        The last feature is the first spectral derivative (per 10 nm), a sensor-agnostic shape cue.
        """
        pooled = F.adaptive_avg_pool2d(x, self.spatial).flatten(2)
        mean, std = foreground_stats(x)
        mean_z, centre, scale = standardise_spectrum(mean, mask)
        d_wl = (wl[:, 1:] - wl[:, :-1]).clamp(min=1e-3)
        deriv = F.pad((mean_z[:, 1:] - mean_z[:, :-1]) / d_wl * 10.0, (0, 1))
        deriv = deriv * (mask & torch.roll(mask, -1, dims=1)).float()
        deriv[:, -1] = 0.0
        feats = torch.cat([(pooled - centre.unsqueeze(-1)) / scale.unsqueeze(-1), mean_z.unsqueeze(-1),
                           (std / scale).unsqueeze(-1), deriv.unsqueeze(-1)], dim=-1)
        return feats * mask.unsqueeze(-1).float()

    def encode(self, x, wl, fwhm, mask, token_mask=None):
        feats = self.band_features(x, wl, mask)
        if self.use_wavelength:
            pos = self.wavelength_embed(wl, fwhm)
        else:
            pos = self.index_embed(torch.arange(x.shape[1], device=x.device)).unsqueeze(0)
            pos = pos.expand(len(x), -1, -1)
        visible = feats if token_mask is None else feats * (~token_mask).unsqueeze(-1).float()
        tokens = self.value_embed(visible) * (1.0 + self.gate(pos)) + pos
        if token_mask is not None:
            tokens = tokens + token_mask.unsqueeze(-1).float() * self.mask_token
        tokens = torch.cat([self.cls_token.expand(len(x), -1, -1), tokens], dim=1)
        pad = torch.cat([torch.zeros(len(x), 1, dtype=torch.bool, device=x.device), ~mask], dim=1)
        return self.norm(self.encoder(tokens, src_key_padding_mask=pad)), feats

    def forward(self, x, wl, fwhm, mask):
        hidden, _ = self.encode(x, wl, fwhm, mask)
        m = mask.unsqueeze(-1).float()
        pooled = (hidden[:, 1:] * m).sum(1) / m.sum(1).clamp(min=1.0)      # mean over real band tokens
        return self.head(torch.cat([hidden[:, 0], pooled], dim=-1))


def pretrain_masked_spectral(model: WST, loader: DataLoader, epochs: int, device) -> float:
    """Mask band tokens at random and regress their spatial summaries. Returns the last epoch loss."""
    opt = torch.optim.AdamW(model.parameters(), lr=CFG["lr"], weight_decay=0.05)
    model.train()
    last = float("nan")
    for _ in range(epochs):
        total, n = 0.0, 0
        for x, wl, fw, mask, _ in loader:
            x, wl, fw, mask = x.to(device), wl.to(device), fw.to(device), mask.to(device)
            token_mask = (torch.rand(mask.shape, device=device) < CFG["mask_ratio"]) & mask
            if not token_mask.any():
                continue
            hidden, feats = model.encode(x, wl, fw, mask, token_mask)
            loss = F.mse_loss(model.reconstruct(hidden[:, 1:])[token_mask], feats[token_mask])
            opt.zero_grad()
            loss.backward()
            opt.step()
            total, n = total + loss.item(), n + 1
        last = total / max(n, 1)
    return last

## 8. Baselines

Fixed-band models cannot ingest cubes with different band grids, so they receive every cube linearly
resampled onto one common wavelength grid (zero outside the measured range). That is the standard
workaround and is the comparison the paper needs.

| Name here | What it is | Label to use in the paper |
|---|---|---|
| `FruitHSNet` | Re-implementation from the description in arXiv:2608.01202 (FFT-magnitude branch + centre-pixel branch, learnable band weights, MLP). The paper's fusion equation is internally inconsistent; the stated 2λ input layer (concatenation) is followed. Official code not verified public. | Fruit-HSNet (re-implemented) |
| `DeepHSNet-Lite` | Compact depth-wise-separable CNN in the spirit of DeepHS-Net; **not** the official network | DeepHS-Net-like |
| `SpectralFormer-Lite` | Group-wise band embedding + transformer on the mean spectrum; simplified | SpectralFormer-Lite |
| `SSFTT-Lite` | 3-D conv → 2-D conv → learned tokeniser → transformer; simplified | SSFTT-Lite |
| `PLS`, `LightGBM` | Classical models on the foreground mean spectrum | PLS-DA / PLS regression, LightGBM |

Official HyveConv++ / DeepHS-Net, MambaHSI and sensor-agnostic foundation models (HyperSL, SpecAware)
need their own repositories and environments; `OPTIONAL_BASELINES` lists where they live. Add them through
`GridAdapter` if you install them.

In [ ]:
def common_grid(cameras: list[str], step_nm: float = 4.0) -> np.ndarray:
    """Shared wavelength grid covering every listed camera."""
    lo = min(CAMERA_GRIDS[c][0] for c in cameras if c in CAMERA_GRIDS) if any(c in CAMERA_GRIDS for c in cameras) else 400.0
    hi = max(CAMERA_GRIDS[c][1] for c in cameras if c in CAMERA_GRIDS) if any(c in CAMERA_GRIDS for c in cameras) else 1000.0
    return np.arange(math.floor(lo), math.ceil(hi) + 1e-6, step_nm, dtype=np.float32)


def resample_to_grid(x, wl, mask, grid):
    """Linear interpolation of padded variable-band cubes onto `grid`; zero outside each sample's range."""
    out = x.new_zeros(x.shape[0], len(grid), *x.shape[2:])
    for i in range(x.shape[0]):
        w, v = wl[i][mask[i]], x[i][mask[i]]
        if len(w) < 2:
            continue
        hi = torch.searchsorted(w, grid).clamp(1, len(w) - 1)
        w0, w1 = w[hi - 1], w[hi]
        t = ((grid - w0) / (w1 - w0).clamp(min=1e-6)).clamp(0, 1)[:, None, None]
        inside = ((grid >= w[0]) & (grid <= w[-1]))[:, None, None]
        out[i] = (v[hi - 1] * (1 - t) + v[hi] * t) * inside
    return out


class GridAdapter(nn.Module):
    """Gives a fixed-band model the same call signature as WST."""

    def __init__(self, inner: nn.Module, grid: np.ndarray):
        super().__init__()
        self.inner = inner
        self.register_buffer("grid", torch.as_tensor(grid, dtype=torch.float32))

    def to_grid(self, x, wl, mask):
        return resample_to_grid(x, wl, mask, self.grid)

    def forward(self, x, wl, fwhm, mask):
        return self.inner(self.to_grid(x, wl, mask))


class FruitHSNet(nn.Module):
    def __init__(self, n_bands: int, n_out: int):
        super().__init__()
        self.w_freq = nn.Parameter(torch.ones(n_bands))
        self.w_spec = nn.Parameter(torch.ones(n_bands))
        self.mlp = nn.Sequential(
            nn.Linear(2 * n_bands, 512), nn.BatchNorm1d(512), nn.ReLU(), nn.Dropout(0.4),
            nn.Linear(512, 256), nn.BatchNorm1d(256), nn.ReLU(), nn.Dropout(0.4),
            nn.Linear(256, n_out))

    def forward(self, x):
        freq = torch.fft.fft2(x).abs().mean((2, 3))                 # spatially averaged FFT magnitude
        centre = x[:, :, x.shape[2] // 2, x.shape[3] // 2]          # centre-pixel spectrum
        return self.mlp(torch.cat([self.w_freq * freq, self.w_spec * centre], dim=1))


class DeepHSNetLite(nn.Module):
    def __init__(self, n_bands: int, n_out: int, width: int = 32):
        super().__init__()

        def block(c_in, c_out):
            return nn.Sequential(nn.Conv2d(c_in, c_in, 3, padding=1, groups=c_in), nn.Conv2d(c_in, c_out, 1),
                                 nn.BatchNorm2d(c_out), nn.ReLU())

        self.features = nn.Sequential(block(n_bands, width), nn.AvgPool2d(2), block(width, width),
                                      nn.AvgPool2d(2), block(width, width))
        self.head = nn.Linear(width, n_out)

    def forward(self, x):
        return self.head(self.features(x).mean((2, 3)))


class SpectralFormerLite(nn.Module):
    def __init__(self, n_bands: int, n_out: int, d_model: int = 64, group: int = 3):
        super().__init__()
        self.group = group
        self.embed = nn.Linear(group, d_model)
        self.pos = nn.Parameter(torch.zeros(1, n_bands, d_model))
        layer = nn.TransformerEncoderLayer(d_model, 4, 2 * d_model, dropout=0.1, batch_first=True)
        self.encoder = nn.TransformerEncoder(layer, 4)
        self.head = nn.Linear(d_model, n_out)

    def forward(self, x):
        spectrum, _ = foreground_stats(x)
        spectrum, _, _ = standardise_spectrum(spectrum, spectrum != 0)      # per-sample standardisation
        groups = F.pad(spectrum, (self.group // 2, self.group // 2)).unfold(1, self.group, 1)
        return self.head(self.encoder(self.embed(groups) + self.pos).mean(1))


class SSFTTLite(nn.Module):
    def __init__(self, n_bands: int, n_out: int, d_model: int = 64, n_tokens: int = 4):
        super().__init__()
        self.conv3d = nn.Sequential(nn.Conv3d(1, 8, 3, padding=1), nn.BatchNorm3d(8), nn.ReLU())
        self.conv2d = nn.Sequential(nn.Conv2d(8 * n_bands, d_model, 3, padding=1), nn.BatchNorm2d(d_model), nn.ReLU())
        self.token_weights = nn.Parameter(torch.randn(1, n_tokens, d_model) * 0.02)
        self.cls_token = nn.Parameter(torch.zeros(1, 1, d_model))
        layer = nn.TransformerEncoderLayer(d_model, 4, 2 * d_model, dropout=0.1, batch_first=True)
        self.encoder = nn.TransformerEncoder(layer, 1)
        self.head = nn.Linear(d_model, n_out)

    def forward(self, x):
        z = self.conv2d(self.conv3d(x.unsqueeze(1)).flatten(1, 2)).flatten(2).transpose(1, 2)   # (N, HW, d)
        attn = torch.softmax(z @ self.token_weights.transpose(1, 2), dim=1)                     # (N, HW, T)
        tokens = attn.transpose(1, 2) @ z                                                       # (N, T, d)
        tokens = torch.cat([self.cls_token.expand(len(z), -1, -1), tokens], dim=1)
        return self.head(self.encoder(tokens)[:, 0])


# name -> settings.  kind: "wst" | fixed-band class | "classical"
MODEL_SPECS = {
    "WST":                 dict(kind="wst", use_wavelength=True, pretrain=True, augment=True),
    "FruitHSNet":          dict(kind=FruitHSNet, augment=True),
    "DeepHSNet-Lite":      dict(kind=DeepHSNetLite, augment=True),
    "SpectralFormer-Lite": dict(kind=SpectralFormerLite, augment=True),
    "SSFTT-Lite":          dict(kind=SSFTTLite, augment=True),
    "PLS":                 dict(kind="classical"),
    "LightGBM":            dict(kind="classical"),
    # ---- ablations of the proposed model ----
    "WST-noWavelength":    dict(kind="wst", use_wavelength=False, pretrain=True, augment=True),
    "WST-noPretrain":      dict(kind="wst", use_wavelength=True, pretrain=False, augment=True),
    "WST-noResampleAug":   dict(kind="wst", use_wavelength=True, pretrain=True, augment=False),
}
MAIN_MODELS = ["WST", "FruitHSNet", "DeepHSNet-Lite", "SpectralFormer-Lite", "SSFTT-Lite", "PLS", "LightGBM"]
ABLATION_MODELS = ["WST-noWavelength", "WST-noPretrain", "WST-noResampleAug"]

OPTIONAL_BASELINES = {
    "HyveConv++ / DeepHS-Net (official)": "https://github.com/cogsys-tuebingen/hyve_conv",
    "ResNet-18 + HyveConv (pretrained weights)": "https://github.com/cogsys-tuebingen/hsi_benchmark",
    "MambaHSI (pins mamba-ssm==1.2.0; Linux + NVIDIA GPU; use a separate environment)":
        "https://github.com/li-yapeng/MambaHSI",
    "HyperSL": "https://github.com/kkweil/HyperSL",
    "SpecAware": "https://github.com/busbyjrj/SpecAware",
}


def build_model(name: str, n_out: int, grid: np.ndarray) -> nn.Module:
    spec = MODEL_SPECS[name]
    if spec["kind"] == "wst":
        return WST(n_out, CFG["d_model"], CFG["n_heads"], CFG["n_layers"], CFG["spatial_tokens"],
                   use_wavelength=spec["use_wavelength"])
    return GridAdapter(spec["kind"](len(grid), n_out), grid)


def count_parameters(model: nn.Module) -> int:
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


_grid = common_grid(list(CAMERA_GRIDS))
pd.Series({n: count_parameters(build_model(n, 3, _grid)) for n in MAIN_MODELS if MODEL_SPECS[n]["kind"] != "classical"},
          name=f"trainable parameters ({len(_grid)}-band grid, 3 classes)")

## 9. Metrics

In [ ]:
from sklearn.metrics import accuracy_score, f1_score, r2_score


def expected_calibration_error(prob: np.ndarray, y: np.ndarray, n_bins: int = 15) -> float:
    conf, pred = prob.max(1), prob.argmax(1)
    edges = np.linspace(0, 1, n_bins + 1)
    ece = 0.0
    for lo, hi in zip(edges[:-1], edges[1:]):
        sel = (conf > lo) & (conf <= hi)
        if sel.any():
            ece += sel.mean() * abs((pred[sel] == y[sel]).mean() - conf[sel].mean())
    return float(ece)


def classification_metrics(prob: np.ndarray, y: np.ndarray) -> dict:
    pred = prob.argmax(1)
    return dict(OA=float(accuracy_score(y, pred)),
                macroF1=float(f1_score(y, pred, average="macro", labels=list(range(prob.shape[1])), zero_division=0)),
                ECE=expected_calibration_error(prob, y))


def regression_metrics(pred: np.ndarray, y: np.ndarray, names: list[str]) -> dict:
    """Targets are standardised with training-set statistics, so MSE = 1 means 'predict the mean'."""
    out = dict(MSE=float(((pred - y) ** 2).mean()))
    r2 = []
    for j, name in enumerate(names):
        out[f"MSE_{name}"] = float(((pred[:, j] - y[:, j]) ** 2).mean())
        out[f"R2_{name}"] = float(r2_score(y[:, j], pred[:, j])) if len(y) > 1 else float("nan")
        r2.append(out[f"R2_{name}"])
    out["R2"] = float(np.mean(r2))
    return out

## 10. Training and evaluation engine

* AdamW with cosine decay; early stopping on the validation loss (only after `min_steps` optimisation
  steps); the best validation checkpoint is restored before testing.
* All neural models share the same optimiser settings, schedule, augmentation and seeds.
* Results are appended to `results.csv` after every model, and finished runs are skipped on restart, so a
  long `FULL_RUN` can be interrupted and resumed.

In [ ]:
class ResultLog:
    """Append-only CSV of result rows with resume support."""

    KEY = ["dataset", "protocol", "fruit", "task", "held_out", "model", "seed"]

    def __init__(self, path: Path):
        self.path = path
        self.rows = pd.read_csv(path).to_dict("records") if path.exists() else []

    def _key(self, row) -> tuple:
        return tuple(str(row.get(k, "")) for k in self.KEY)

    def done(self, **meta) -> bool:
        key = self._key(meta)
        return any(self._key(r) == key for r in self.rows)

    def add(self, rows: list[dict]) -> None:
        self.rows.extend(rows)
        pd.DataFrame(self.rows).to_csv(self.path, index=False)

    def frame(self) -> pd.DataFrame:
        return pd.DataFrame(self.rows)


LOG = ResultLog(OUT_DIR / "results.csv")


def _to(batch, device):
    return [t.to(device) for t in batch]


@torch.no_grad()
def predict(model: nn.Module, ds: Dataset, device, regression: bool):
    """Returns (class probabilities or regression outputs, targets) as NumPy arrays."""
    model.eval()
    outs, ys = [], []
    for batch in make_loader(ds, False, 0):
        x, wl, fw, mask, y = _to(batch, device)
        out = model(x, wl, fw, mask)
        outs.append((out if regression else out.softmax(-1)).cpu())
        ys.append(y.cpu())
    return torch.cat(outs).numpy(), torch.cat(ys).numpy()


def train_neural(name: str, n_out: int, grid, train_ds, val_ds, seed: int, regression: bool,
                 device=None, epochs: int | None = None, pretrain_ds=None):
    """Train one neural model. Returns (model, info dict)."""
    device = device or DEVICE
    epochs = epochs or CFG["epochs"]
    set_seed(seed)
    model = build_model(name, n_out, grid).to(device)
    if device.type == "cuda":
        torch.cuda.reset_peak_memory_stats()
    t0 = time.perf_counter()
    pretrain_loss = float("nan")
    if MODEL_SPECS[name].get("pretrain") and pretrain_ds is not None and len(pretrain_ds) > 1:
        pretrain_loss = pretrain_masked_spectral(model, make_loader(pretrain_ds, True, seed),
                                                 CFG["pretrain_epochs"], device)
    pretrain_s = time.perf_counter() - t0

    opt = torch.optim.AdamW(model.parameters(), lr=CFG["lr"], weight_decay=CFG["weight_decay"])
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    loss_fn = nn.MSELoss() if regression else nn.CrossEntropyLoss()
    loader = make_loader(train_ds, True, seed)
    best_loss, best_state, bad_epochs, ran, steps = float("inf"), None, 0, 0, 0
    for epoch in range(epochs):
        model.train()
        for batch in loader:
            x, wl, fw, mask, y = _to(batch, device)
            loss = loss_fn(model(x, wl, fw, mask), y)
            opt.zero_grad()
            loss.backward()
            opt.step()
            steps += 1
        sched.step()
        ran = epoch + 1
        if val_ds is not None and len(val_ds):
            out, yv = predict(model, val_ds, device, regression)
            val_loss = (float(((out - yv) ** 2).mean()) if regression
                        else float(-np.log(out[np.arange(len(yv)), yv] + 1e-9).mean()))
            if val_loss < best_loss - 1e-6:
                best_loss, bad_epochs = val_loss, 0
                best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
            else:
                bad_epochs += 1
                # Small datasets give few steps per epoch and the loss can sit on a plateau first,
                # so early stopping is only allowed after a minimum number of optimisation steps.
                if bad_epochs > CFG["patience"] and steps >= CFG["min_steps"]:
                    break
    if best_state is not None:
        model.load_state_dict(best_state)
    info = dict(train_s=time.perf_counter() - t0, pretrain_s=pretrain_s, pretrain_loss=pretrain_loss,
                epochs_run=ran, steps_run=steps, n_params=count_parameters(model), device=device.type,
                peak_vram_mb=torch.cuda.max_memory_allocated() / 2 ** 20 if device.type == "cuda" else float("nan"))
    return model, info


# ---- classical models on foreground mean spectra ----
def mean_spectra(ds: Dataset, grid: np.ndarray):
    """Foreground mean spectrum of every cube, resampled onto `grid`. Returns (X, y)."""
    g = torch.as_tensor(grid, dtype=torch.float32)
    feats, ys = [], []
    for x, wl, fw, mask, y in make_loader(ds, False, 0):
        mean, _ = foreground_stats(x)
        feats.append(resample_to_grid(mean[:, :, None, None], wl, mask, g)[:, :, 0, 0])
        ys.append(y)
    return torch.cat(feats).numpy(), torch.cat(ys).numpy()


class ClassicalModel:
    """PLS-DA / PLS regression or LightGBM with a predict() that mirrors the neural models."""

    def __init__(self, name: str, n_out: int, regression: bool, seed: int):
        self.name, self.n_out, self.regression, self.seed = name, n_out, regression, seed

    def fit(self, X, y):
        from sklearn.cross_decomposition import PLSRegression
        from lightgbm import LGBMClassifier, LGBMRegressor
        if self.name == "PLS":
            target = y if self.regression else np.eye(self.n_out)[y]
            n_comp = int(max(1, min(10, X.shape[1], len(X) - 1)))
            self.model = PLSRegression(n_components=n_comp).fit(X, target)
        else:
            kw = dict(n_estimators=200, learning_rate=0.05, num_leaves=15, min_child_samples=2,
                      subsample=0.8, subsample_freq=1, colsample_bytree=0.5, random_state=self.seed,
                      verbose=-1, n_jobs=psutil.cpu_count(logical=False) or 1)
            if self.regression:
                self.model = [LGBMRegressor(**kw).fit(X, y[:, j]) for j in range(self.n_out)]
            else:
                self.model = LGBMClassifier(**kw).fit(X, y)
        return self

    def predict(self, X):
        if self.name == "PLS":
            out = np.asarray(self.model.predict(X)).reshape(len(X), -1)
            if self.regression:
                return out
            e = np.exp(out - out.max(1, keepdims=True))            # scores -> pseudo-probabilities
            return e / e.sum(1, keepdims=True)
        if self.regression:
            return np.stack([m.predict(X) for m in self.model], axis=1)
        prob = np.zeros((len(X), self.n_out))
        prob[:, self.model.classes_] = self.model.predict_proba(X)
        return prob


def run_models(models, *, hw, grid, n_out, regression, y_cols, target_names, train_df, val_df, tests,
               pretrain_df, seed, meta, log=LOG):
    """Train every model on one split and score it on every test set.

    tests: list of (tag, dataframe, wavelength_subset_or_None). One result row per (model, tag).
    """
    def score(out, y):
        return regression_metrics(out, y, target_names) if regression else classification_metrics(out, y)

    def val_score(out, y):                          # higher is better; used only for model selection
        return -float(((out - y) ** 2).mean()) if regression else classification_metrics(out, y)["macroF1"]

    for name in models:
        key = dict(meta, model=name, seed=seed)
        if log.done(**key):
            continue
        spec = MODEL_SPECS[name]
        plain = lambda df, subset=None: CubeDataset(df, hw, y_cols, subset=subset)
        if spec["kind"] == "classical":
            t0 = time.perf_counter()
            model = ClassicalModel(name, n_out, regression, seed).fit(*mean_spectra(plain(train_df), grid))
            info = dict(train_s=time.perf_counter() - t0, device="cpu")
            infer = lambda ds: (model.predict(mean_spectra(ds, grid)[0]), mean_spectra(ds, grid)[1])
        else:
            train_ds = CubeDataset(train_df, hw, y_cols, augment=spec.get("augment", False), seed=seed)
            pre_ds = (CubeDataset(pretrain_df, hw, None, augment=spec.get("augment", False), seed=seed)
                      if pretrain_df is not None else None)
            model, info = train_neural(name, n_out, grid, train_ds, plain(val_df), seed, regression,
                                       pretrain_ds=pre_ds)
            infer = lambda ds: predict(model, ds, DEVICE, regression)
        vs = val_score(*infer(plain(val_df))) if len(val_df) else float("nan")
        rows = []
        for tag, test_df, subset in tests:
            out, y = infer(plain(test_df, subset))
            rows.append(dict(key, data_source=DATA_SOURCE, test_set=tag, n_train=len(train_df),
                             n_test=len(test_df), val_score=vs, **score(out, y), **info))
        log.add(rows)

## 11. Experiment A — DeepHS Fruit: leave-one-camera-out, within-camera reference, day-disjoint

For each fruit × task × seed × held-out camera:

* `LOCO`: every main model and (optionally) the WST ablations;
* `WITHIN`: every main model, trained on the held-out camera and scored on the same test recordings;
* `DAY`: every main model, once per fruit × task × seed.

WST is pretrained only on recordings from the training cameras and training specimens of the fold.

In [ ]:
def encode_classes(df: pd.DataFrame, task: str):
    """Add an integer class column '_y' for `task`. Class order follows CLASS_NAMES when the strings match."""
    present = sorted(df[task].dropna().unique().tolist())
    canonical = [c for c in CLASS_NAMES.get(task, []) if c in present]
    classes = canonical if len(canonical) == len(present) else present
    out = df.copy()
    out["_y"] = out[task].map({c: i for i, c in enumerate(classes)})
    return out, classes


def run_deephs() -> None:
    for fruit in CFG["fruits"]:
        cams = sorted(DEEPHS.loc[DEEPHS["fruit"] == fruit, "camera"].unique())
        grid = common_grid(cams)
        for task in CFG["tasks"]:
            labelled = DEEPHS[(DEEPHS["fruit"] == fruit) & DEEPHS[task].notna()]
            if labelled[task].nunique() < 2 or labelled[task].nunique() > 10:
                print(f"skip {fruit}/{task}: no usable class labels")
                continue
            data, classes = encode_classes(DEEPHS, task)
            common = dict(hw=CFG["deephs_hw"], grid=grid, n_out=len(classes), regression=False,
                          y_cols="_y", target_names=None)
            for seed in SEEDS:
                for fold in loco_folds(data, fruit, task, seed):
                    meta = dict(dataset="DeepHS", fruit=fruit, task=task, held_out=fold["held_out"])
                    tests = [("test", fold["test"], None)]
                    if "LOCO" in CFG["protocols"]:
                        models = MAIN_MODELS + (ABLATION_MODELS if CFG["run_ablations"] else [])
                        run_models(models, **common, train_df=fold["loco_train"], val_df=fold["loco_val"],
                                   tests=tests, pretrain_df=fold["loco_pretrain"], seed=seed,
                                   meta=dict(meta, protocol="LOCO", tier=fold["tier"], coverage=fold["coverage"]))
                    if "WITHIN" in CFG["protocols"]:
                        run_models(MAIN_MODELS, **common, train_df=fold["within_train"],
                                   val_df=fold["within_val"], tests=tests,
                                   pretrain_df=fold["within_pretrain"], seed=seed,
                                   meta=dict(meta, protocol="WITHIN", tier=fold["tier"], coverage=fold["coverage"]))
                if "DAY" in CFG["protocols"]:
                    fold = day_disjoint_fold(data, fruit, task, seed)
                    if fold is not None:
                        free = data[(data["fruit"] == fruit)
                                    & ~data["fruit_id"].isin(set(fold["test"]["fruit_id"]) | set(fold["val"]["fruit_id"]))]
                        run_models(MAIN_MODELS, **common, train_df=fold["train"], val_df=fold["val"],
                                   tests=[("test", fold["test"], None)], pretrain_df=free, seed=seed,
                                   meta=dict(dataset="DeepHS", fruit=fruit, task=task, held_out="-",
                                             protocol="DAY", tier="mixed"))
                print(f"done: {fruit} / {task} / seed {seed}")


_t0 = time.perf_counter()
run_deephs()
print(f"Experiment A finished in {(time.perf_counter() - _t0) / 60:.1f} min; {len(LOG.rows)} result rows so far")

## 12. Experiment B — HyperLeaf2024: plot-level split and band-subset shift

HyperLeaf2024 has a single camera, so it cannot test camera shift. It tests **wavelength-subset shift**:
models are trained on the full 400–1000 nm range and scored on the full range and on restricted ranges
without retraining. Two tasks are run: regression of the four standardised traits, and cultivar
classification.

Fixed-band baselines see zeros outside the available range, which is what happens when such a model is
deployed on a sensor with a narrower range. Linear models (PLS) can then fail by orders of magnitude;
that is a property of the method, not a bug, and the figure uses a logarithmic axis for that reason.

In [ ]:
LEAF_SUBSETS = {"full": None, "VIS_400_700": (400.0, 700.0), "NIR_700_1000": (700.0, 1000.0),
                "trimmed_450_950": (450.0, 950.0)}


def run_hyperleaf() -> None:
    grid = np.arange(400.0, 1000.0 + 1e-6, 4.0, dtype=np.float32)
    for seed in SEEDS:
        split = plot_level_split(HLEAF, seed)
        # ---- trait regression: standardise targets with TRAINING statistics only ----
        mu, sd = split["train"][LEAF_TARGETS].mean(), split["train"][LEAF_TARGETS].std().replace(0, 1.0)
        z_cols = [f"_z_{t}" for t in LEAF_TARGETS]
        parts = {}
        for k, part in split.items():
            part = part.copy()
            part[z_cols] = ((part[LEAF_TARGETS] - mu) / sd).to_numpy()
            parts[k] = part
        tests = [(tag, parts["test"], subset) for tag, subset in LEAF_SUBSETS.items()]
        run_models(MAIN_MODELS, hw=CFG["leaf_hw"], grid=grid, n_out=len(LEAF_TARGETS), regression=True,
                   y_cols=z_cols, target_names=LEAF_TARGETS, train_df=parts["train"], val_df=parts["val"],
                   tests=tests, pretrain_df=parts["train"], seed=seed,
                   meta=dict(dataset="HyperLeaf", fruit="wheat", task="traits", held_out="-",
                             protocol="PLOT", tier="single-camera"))
        # ---- cultivar classification ----
        if HLEAF["cultivar"].notna().all() and HLEAF["cultivar"].nunique() > 1:
            classes = sorted(HLEAF["cultivar"].unique())
            parts = {k: part.assign(_y=part["cultivar"].map({c: i for i, c in enumerate(classes)}))
                     for k, part in split.items()}
            tests = [(tag, parts["test"], subset) for tag, subset in LEAF_SUBSETS.items()]
            run_models(MAIN_MODELS, hw=CFG["leaf_hw"], grid=grid, n_out=len(classes), regression=False,
                       y_cols="_y", target_names=None, train_df=parts["train"], val_df=parts["val"],
                       tests=tests, pretrain_df=parts["train"], seed=seed,
                       meta=dict(dataset="HyperLeaf", fruit="wheat", task="cultivar", held_out="-",
                                 protocol="PLOT", tier="single-camera"))
        print(f"done: HyperLeaf / seed {seed}")


_t0 = time.perf_counter()
run_hyperleaf()
print(f"Experiment B finished in {(time.perf_counter() - _t0) / 60:.1f} min; {len(LOG.rows)} result rows in total")

## 13. Statistical analysis

* **Blocks** are configurations (fruit × task × held-out camera). Seeds of one configuration are not
  independent, so scores are **averaged over seeds first**; this is the conservative primary analysis.
* **Friedman test** across all main models, with **Nemenyi** post-hoc comparisons.
* **Wilcoxon signed-rank test** of WST against each baseline, **Holm**-corrected. The "strongest
  baseline" is chosen by mean *validation* score, never by test score.
* With few configurations these tests have little power; report the number of blocks beside every p-value.

In [ ]:
from scipy.stats import friedmanchisquare, wilcoxon
import scikit_posthocs as sp

CONFIG_COLS = ["dataset", "protocol", "fruit", "task", "held_out", "test_set"]


def seed_averaged(df: pd.DataFrame, metric: str, models: list[str]) -> pd.DataFrame:
    """Rows = configurations, columns = models, values = mean over seeds."""
    d = df[df["model"].isin(models)]
    table = d.pivot_table(index=CONFIG_COLS, columns="model", values=metric, aggfunc="mean")
    return table.reindex(columns=[m for m in models if m in table.columns]).dropna()


def holm(pvalues: dict) -> dict:
    """Holm step-down adjustment."""
    items = sorted(pvalues.items(), key=lambda kv: kv[1])
    adjusted, running = {}, 0.0
    for rank, (name, p) in enumerate(items):
        running = max(running, min(1.0, (len(items) - rank) * p))
        adjusted[name] = running
    return adjusted


def statistical_report(df: pd.DataFrame, metric: str = "macroF1", protocol: str = "LOCO",
                       proposed: str = "WST", higher_is_better: bool = True, tier: str | None = None):
    d = df[(df["protocol"] == protocol)]
    if tier is not None:
        d = d[d["tier"] == tier]
    table = seed_averaged(d, metric, MAIN_MODELS)
    tag = f"{protocol}_{metric}" + (f"_{tier}" if tier else "")
    report = dict(metric=metric, protocol=protocol, tier=tier or "all", n_blocks=int(len(table)),
                  data_source=DATA_SOURCE)
    if len(table) < 2 or table.shape[1] < 3 or proposed not in table:
        report["note"] = "not enough configurations or models for a test"
        return report, None
    ranks = table.rank(axis=1, ascending=not higher_is_better)
    report["mean_ranks"] = ranks.mean().round(3).to_dict()
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", RuntimeWarning)
        stat, p = friedmanchisquare(*[table[c].to_numpy() for c in table.columns])
    if np.isfinite(stat):
        report.update(friedman_chi2=float(stat), friedman_p=float(p))
    else:
        report["friedman_note"] = "undefined: every model is tied in every configuration"
    nemenyi = sp.posthoc_nemenyi_friedman(table.to_numpy() * (1 if higher_is_better else -1))
    nemenyi.index = nemenyi.columns = table.columns
    nemenyi.to_csv(OUT_DIR / f"nemenyi_{tag}.csv")
    # Wilcoxon: proposed vs every baseline, Holm-corrected
    raw = {}
    for base in table.columns.drop(proposed):
        diff = table[proposed] - table[base]
        raw[base] = float(wilcoxon(diff, zero_method="pratt").pvalue) if (diff != 0).any() else 1.0
    report["wilcoxon_p_raw"] = raw
    report["wilcoxon_p_holm"] = holm(raw)
    # strongest baseline by validation score
    val = d[d["model"].isin(table.columns.drop(proposed))].groupby("model")["val_score"].mean()
    if val.notna().any():
        report["strongest_baseline_by_validation"] = str(val.idxmax())
    (OUT_DIR / f"stats_{tag}.json").write_text(json.dumps(report, indent=2))
    return report, nemenyi


RESULTS = LOG.frame()
STATS = {}
for _metric in ("macroF1", "OA"):
    STATS[_metric], _nemenyi = statistical_report(RESULTS, _metric, "LOCO")
for _tier in ("overlap", "extrapolation"):
    STATS[f"macroF1_{_tier}"], _ = statistical_report(RESULTS, "macroF1", "LOCO", tier=_tier)
print(json.dumps(STATS["macroF1"], indent=2))

## 14. CPU versus GPU efficiency study

For each neural model, on one representative LOCO fold:

| Quantity | How it is measured |
|---|---|
| Parameters | trainable parameter count |
| FLOPs | `fvcore` on one cube; operators that fvcore cannot count are skipped, so treat it as a **lower bound** |
| Training time, peak memory, energy | a fixed number of epochs on CPU and (if present) GPU |
| Inference latency | batch size 1, median and 95th percentile after warm-up |
| Throughput | cubes per second at a fixed batch size |
| INT8 on CPU | dynamic quantisation of linear layers: latency **and** accuracy against FP32 |
| Energy | CodeCarbon (`OfflineEmissionsTracker`, process mode). It reads RAPL counters on Linux and NVML on NVIDIA GPUs and **falls back to a TDP-based estimate** when the counters are unreadable; the column `energy_source` records which applies. These are estimates, not wall-meter readings. |

Latency of the fixed-band baselines includes resampling onto the common grid, because that step is part of
deploying them on a new camera.

In [ ]:
class EnergyMeter:
    """Context manager around CodeCarbon. Yields NaN when CodeCarbon is unavailable."""

    def __enter__(self):
        self.kwh = self.cpu_kwh = self.gpu_kwh = float("nan")
        self.t0 = time.perf_counter()
        try:
            import logging
            from codecarbon import OfflineEmissionsTracker
            logging.getLogger("codecarbon").setLevel(logging.ERROR)
            self.tracker = OfflineEmissionsTracker(
                country_iso_code=CFG["country_iso"], measure_power_secs=1, tracking_mode="process",
                save_to_file=False, log_level="error", allow_multiple_runs=True)
            self.tracker.start()
        except Exception as err:                                   # noqa: BLE001
            warnings.warn(f"CodeCarbon unavailable: {err}")
            self.tracker = None
        return self

    def __exit__(self, *exc):
        self.seconds = time.perf_counter() - self.t0
        if self.tracker is not None:
            try:
                self.tracker.stop()
                data = self.tracker.final_emissions_data
                self.kwh, self.cpu_kwh, self.gpu_kwh = data.energy_consumed, data.cpu_energy, data.gpu_energy
            except Exception as err:                               # noqa: BLE001
                warnings.warn(f"CodeCarbon failed: {err}")
        return False

    @property
    def joules(self) -> float:
        return self.kwh * 3.6e6


def energy_source() -> str:
    rapl = Path("/sys/class/powercap/intel-rapl")
    cpu = "RAPL" if rapl.exists() and any(os.access(p, os.R_OK) for p in rapl.glob("*/energy_uj")) else "TDP-estimate"
    return f"cpu:{cpu}" + (", gpu:NVML" if torch.cuda.is_available() else "")


def count_flops(model: nn.Module, batch) -> float:
    try:
        import logging
        from fvcore.nn import FlopCountAnalysis
        logging.getLogger("fvcore").setLevel(logging.ERROR)
        x, wl, fw, mask, _ = [t.cpu() for t in batch]
        m = model.cpu().eval()
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            if isinstance(m, GridAdapter):
                analysis = FlopCountAnalysis(m.inner, (m.to_grid(x, wl, mask),))
            else:
                analysis = FlopCountAnalysis(m, (x, wl, fw, mask))
            analysis.unsupported_ops_warnings(False).uncalled_modules_warnings(False)
            return float(analysis.total())
    except Exception as err:                                       # noqa: BLE001
        warnings.warn(f"FLOP count unavailable: {err}")
        return float("nan")


@torch.inference_mode()
def measure_latency(model: nn.Module, batch, device, repeats: int) -> dict:
    """Median / p95 latency (ms) for the given batch, with warm-up and CUDA synchronisation."""
    model = model.to(device).eval()
    x, wl, fw, mask, _ = _to(batch, device)
    sync = torch.cuda.synchronize if device.type == "cuda" else (lambda: None)
    for _ in range(5):
        model(x, wl, fw, mask)
    sync()
    times = []
    for _ in range(repeats):
        t = time.perf_counter()
        model(x, wl, fw, mask)
        sync()
        times.append((time.perf_counter() - t) * 1e3)
    return dict(median_ms=float(np.median(times)), p95_ms=float(np.percentile(times, 95)))


@torch.inference_mode()
def measure_inference_energy(model: nn.Module, batch, device) -> dict:
    """Energy per cube at batch size 1, measured over a loop long enough for the power sampler."""
    model = model.to(device).eval()
    x, wl, fw, mask, _ = _to(batch, device)
    n = 0
    with EnergyMeter() as meter:
        while time.perf_counter() - meter.t0 < CFG["energy_min_seconds"]:
            model(x, wl, fw, mask)
            n += 1
        if device.type == "cuda":
            torch.cuda.synchronize()
    return dict(joules_per_cube=meter.joules / max(n, 1), n_inferences=n)


def quantise_int8(model: nn.Module):
    """Dynamic INT8 quantisation of linear layers for CPU inference. Returns (model, backend)."""
    import copy
    m = copy.deepcopy(model).cpu().eval()
    try:
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            return torch.ao.quantization.quantize_dynamic(m, {nn.Linear}, dtype=torch.qint8), "torch.ao dynamic"
    except Exception:                                              # noqa: BLE001  (API removed in newer PyTorch)
        from torchao.quantization import Int8DynamicActivationInt8WeightConfig, quantize_
        quantize_(m, Int8DynamicActivationInt8WeightConfig())
        return m, "torchao"


def efficiency_study() -> pd.DataFrame:
    """Train each neural model briefly on CPU and GPU for one fold, then profile inference."""
    fruit, task = CFG["fruits"][0], CFG["tasks"][0]
    data, classes = encode_classes(DEEPHS, task)
    fold = loco_folds(data, fruit, task, seed=0)[0]
    grid = common_grid(sorted(data.loc[data["fruit"] == fruit, "camera"].unique()))
    hw = CFG["deephs_hw"]
    test_ds = CubeDataset(fold["test"], hw, "_y")
    one = next(iter(make_loader(test_ds, False, 0, batch=1)))
    many = next(iter(make_loader(test_ds, False, 0, batch=min(CFG["throughput_batch"], len(test_ds)))))
    devices = [torch.device("cpu")] + ([torch.device("cuda")] if torch.cuda.is_available() else [])
    cpu = torch.device("cpu")
    rows = []
    for name in [m for m in MAIN_MODELS if MODEL_SPECS[m]["kind"] != "classical"]:
        row = dict(model=name, data_source=DATA_SOURCE, fruit=fruit, held_out=fold["held_out"],
                   bands_in=int(one[3].sum()), hw=f"{hw[0]}x{hw[1]}", energy_source=energy_source(),
                   torch_threads=torch.get_num_threads())
        model = None
        for dev in devices:
            train_ds = CubeDataset(fold["loco_train"], hw, "_y", augment=True, seed=0)
            pre_ds = CubeDataset(fold["loco_pretrain"], hw, None, augment=True, seed=0)
            with EnergyMeter() as meter:
                model, info = train_neural(name, len(classes), grid, train_ds, CubeDataset(fold["loco_val"], hw, "_y"),
                                           0, False, device=dev, epochs=CFG["eff_epochs"], pretrain_ds=pre_ds)
            d = dev.type
            row.update({f"train_s_{d}": info["train_s"], f"train_kJ_{d}": meter.joules / 1e3,
                        "n_params": info["n_params"], "train_epochs": info["epochs_run"]})
            if d == "cuda":
                row["train_peak_vram_mb"] = info["peak_vram_mb"]
            lat = measure_latency(model, one, dev, CFG["eff_repeats"])
            thr = measure_latency(model, many, dev, max(5, CFG["eff_repeats"] // 5))
            row.update({f"latency_ms_{d}": lat["median_ms"], f"latency_p95_ms_{d}": lat["p95_ms"],
                        f"throughput_cubes_s_{d}": len(many[0]) / thr["median_ms"] * 1e3,
                        f"joules_per_cube_{d}": measure_inference_energy(model, one, dev)["joules_per_cube"]})
            if d == "cuda":
                torch.cuda.reset_peak_memory_stats()
                measure_latency(model, many, dev, 3)
                row["infer_peak_vram_mb"] = torch.cuda.max_memory_allocated() / 2 ** 20
        row["flops_per_cube"] = count_flops(model, one)
        # ---- FP32 vs INT8 on CPU: accuracy and latency ----
        model = model.to(cpu).eval()
        prob, y = predict(model, test_ds, cpu, False)
        row["macroF1_fp32"] = classification_metrics(prob, y)["macroF1"]
        try:
            q_model, backend = quantise_int8(model)
            prob_q, _ = predict(q_model, test_ds, cpu, False)
            row.update(int8_backend=backend, macroF1_int8=classification_metrics(prob_q, y)["macroF1"],
                       latency_ms_cpu_int8=measure_latency(q_model, one, cpu, CFG["eff_repeats"])["median_ms"])
        except Exception as err:                                   # noqa: BLE001
            row.update(int8_backend=f"failed: {type(err).__name__}")
        row["rss_mb"] = psutil.Process().memory_info().rss / 2 ** 20
        rows.append(row)
        print("profiled", name)
    table = pd.DataFrame(rows)
    table.to_csv(OUT_DIR / "efficiency.csv", index=False)
    return table


EFFICIENCY = efficiency_study()
EFFICIENCY.set_index("model").T

## 15. Tables and figures

Every table is written as CSV and LaTeX, every figure as PDF and PNG, in the results folder.
Mean ± standard deviation is taken **across seeds**, after averaging over configurations within a seed.

In [ ]:
import matplotlib.pyplot as plt

plt.rcParams.update({"font.family": "serif", "font.size": 9, "axes.spines.top": False,
                     "axes.spines.right": False, "figure.dpi": 120, "savefig.dpi": 300,
                     "savefig.bbox": "tight"})
PALETTE = ["#0072B2", "#E69F00", "#009E73", "#CC79A7", "#56B4E9", "#D55E00", "#999999",
           "#F0E442", "#000000", "#7F3C8D"]                       # colour-blind-safe (Okabe–Ito based)
COLOUR = {m: PALETTE[i % len(PALETTE)] for i, m in enumerate(MODEL_SPECS)}


def mean_sd_table(df: pd.DataFrame, by: list[str], metrics: list[str]) -> pd.DataFrame:
    """Mean ± s.d. across seeds of the per-seed mean over configurations."""
    per_seed = df.groupby([*by, "seed"], dropna=False)[metrics].mean()
    stats = per_seed.groupby(by, dropna=False).agg(["mean", "std"])
    text = pd.DataFrame({m: [f"{a:.3f} ± {0.0 if pd.isna(b) else b:.3f}"
                             for a, b in zip(stats[(m, "mean")], stats[(m, "std")])] for m in metrics},
                        index=stats.index)
    text["n_seeds"] = per_seed.groupby(by, dropna=False).size().to_numpy()
    return text


def to_latex(table: pd.DataFrame, caption: str, label: str) -> str:
    t = table.reset_index()
    esc = lambda s: str(s).replace("_", r"\_").replace("±", r"$\pm$").replace("%", r"\%")
    lines = [r"\begin{table}[t]", r"\centering", rf"\caption{{{esc(caption)}}}", rf"\label{{{label}}}",
             r"\begin{tabular}{" + "l" * len(t.columns) + "}", r"\hline",
             " & ".join(esc(c) for c in t.columns) + r" \\", r"\hline"]
    lines += [" & ".join(esc(v) for v in row) + r" \\" for row in t.itertuples(index=False)]
    lines += [r"\hline", r"\end{tabular}", r"\end{table}"]
    return "\n".join(lines)


def save_table(table: pd.DataFrame, name: str, caption: str) -> pd.DataFrame:
    table.to_csv(OUT_DIR / f"{name}.csv")
    (OUT_DIR / f"{name}.tex").write_text(to_latex(table, caption, f"tab:{name}"))
    return table


def save_fig(fig, name: str) -> None:
    fig.savefig(FIG_DIR / f"{name}.pdf")
    fig.savefig(FIG_DIR / f"{name}.png")
    plt.show()


def per_camera_drop(df: pd.DataFrame, metric: str = "macroF1") -> pd.DataFrame:
    """WITHIN − LOCO on identical test recordings; one row per fruit, task, camera, model and seed."""
    keys = ["fruit", "task", "held_out", "tier", "model", "seed"]
    within = df[df["protocol"] == "WITHIN"].set_index(keys)[metric]
    loco = df[df["protocol"] == "LOCO"].set_index(keys)[metric]
    out = pd.concat([within.rename("within"), loco.rename("loco")], axis=1).dropna()
    out["drop"] = out["within"] - out["loco"]
    return out.reset_index()


RESULTS = LOG.frame()
SYN_NOTE = " SYNTHETIC SMOKE-TEST DATA: NOT A RESULT." if USE_SYNTHETIC else ""
fruit_df = RESULTS[RESULTS["dataset"] == "DeepHS"]
main_df = fruit_df[fruit_df["model"].isin(MAIN_MODELS)]

T_MAIN = save_table(mean_sd_table(main_df, ["protocol", "model"], ["OA", "macroF1", "ECE"]),
                    "table_deephs_main", "DeepHS Fruit: mean ± s.d. over seeds." + SYN_NOTE)
T_TIER = save_table(mean_sd_table(main_df[main_df["protocol"] == "LOCO"], ["tier", "model"], ["OA", "macroF1", "ECE"]),
                    "table_deephs_loco_by_tier", "Leave-one-camera-out by spectral tier." + SYN_NOTE)
DROP = per_camera_drop(main_df)
T_DROP = save_table(mean_sd_table(DROP, ["held_out", "tier", "model"], ["within", "loco", "drop"]),
                    "table_per_camera_drop", "Per-camera drop in macro-F1 (within − LOCO)." + SYN_NOTE)
if CFG["run_ablations"]:
    abl = fruit_df[(fruit_df["protocol"] == "LOCO") & fruit_df["model"].isin(["WST", *ABLATION_MODELS])]
    T_ABL = save_table(mean_sd_table(abl, ["model"], ["OA", "macroF1", "ECE"]),
                       "table_ablation", "Ablation of WST under leave-one-camera-out." + SYN_NOTE)
leaf_df = RESULTS[RESULTS["dataset"] == "HyperLeaf"]
T_LEAF_REG = save_table(mean_sd_table(leaf_df[leaf_df["task"] == "traits"], ["test_set", "model"], ["MSE", "R2"]),
                        "table_hyperleaf_traits", "HyperLeaf2024 traits: standardised MSE and mean R²." + SYN_NOTE)
if (leaf_df["task"] == "cultivar").any():
    T_LEAF_CLS = save_table(mean_sd_table(leaf_df[leaf_df["task"] == "cultivar"], ["test_set", "model"], ["OA", "macroF1"]),
                            "table_hyperleaf_cultivar", "HyperLeaf2024 cultivar classification." + SYN_NOTE)
print("Table: DeepHS Fruit main results" + SYN_NOTE)
T_MAIN

In [ ]:
def fig_loco_vs_within(df: pd.DataFrame) -> None:
    per_seed = df.groupby(["protocol", "model", "seed"])["macroF1"].mean()
    stats = per_seed.groupby(["protocol", "model"]).agg(["mean", "std"]).fillna(0)
    models = [m for m in MAIN_MODELS if ("LOCO", m) in stats.index]
    fig, ax = plt.subplots(figsize=(6.4, 2.8))
    x, width = np.arange(len(models)), 0.38
    for i, (protocol, hatch) in enumerate((("WITHIN", ""), ("LOCO", "//"))):
        vals = [stats.loc[(protocol, m), "mean"] if (protocol, m) in stats.index else np.nan for m in models]
        errs = [stats.loc[(protocol, m), "std"] if (protocol, m) in stats.index else 0 for m in models]
        ax.bar(x + (i - 0.5) * width, vals, width, yerr=errs, capsize=2, hatch=hatch, edgecolor="white",
               color=[COLOUR[m] for m in models], alpha=1.0 if protocol == "LOCO" else 0.55,
               label="within-camera" if protocol == "WITHIN" else "leave-one-camera-out")
    ax.set_xticks(x, models, rotation=25, ha="right")
    ax.set_ylabel("Macro-F1")
    ax.legend(frameon=False, fontsize=7)
    ax.set_title("Within-camera versus leave-one-camera-out" + (" [synthetic smoke test]" if USE_SYNTHETIC else ""), fontsize=8)
    save_fig(fig, "fig_loco_vs_within")


def fig_camera_drop(drop: pd.DataFrame) -> None:
    cams = sorted(drop["held_out"].unique())
    models = [m for m in MAIN_MODELS if m in set(drop["model"])]
    fig, ax = plt.subplots(figsize=(6.4, 2.8))
    width = 0.8 / max(len(models), 1)
    for j, m in enumerate(models):
        d = drop[drop["model"] == m].groupby("held_out")["drop"].agg(["mean", "std"]).reindex(cams).fillna(0)
        ax.bar(np.arange(len(cams)) + j * width, d["mean"], width, yerr=d["std"], capsize=2,
               color=COLOUR[m], label=m)
    ax.axhline(0, color="black", lw=0.6)
    ax.set_xticks(np.arange(len(cams)) + 0.4 - width / 2, cams)
    ax.set_xlabel("Held-out camera")
    ax.set_ylabel("Macro-F1 drop (within − LOCO)")
    ax.legend(frameon=False, fontsize=6, ncol=4, loc="upper center", bbox_to_anchor=(0.5, 1.3))
    save_fig(fig, "fig_per_camera_drop")


def fig_band_subset(df: pd.DataFrame, task: str, metric: str, ylabel: str) -> None:
    d = df[df["task"] == task]
    if d.empty:
        return
    per_seed = d.groupby(["test_set", "model", "seed"])[metric].mean()
    stats = per_seed.groupby(["test_set", "model"]).agg(["mean", "std"]).fillna(0)
    subsets = [s for s in LEAF_SUBSETS if s in stats.index.get_level_values(0)]
    fig, ax = plt.subplots(figsize=(5.2, 2.8))
    for m in [m for m in MAIN_MODELS if m in set(d["model"])]:
        vals = [stats.loc[(s, m), "mean"] for s in subsets]
        errs = [stats.loc[(s, m), "std"] for s in subsets]
        ax.errorbar(range(len(subsets)), vals, yerr=errs, marker="o", ms=3, lw=1, capsize=2,
                    color=COLOUR[m], label=m)
    ax.set_xticks(range(len(subsets)), subsets, rotation=15)
    ax.set_xlabel("Bands available at test time (trained on full range)")
    ax.set_ylabel(ylabel)
    if metric == "MSE":
        ax.set_yscale("log")                     # fixed-band models can fail by orders of magnitude
    ax.legend(frameon=False, fontsize=6, ncol=2)
    save_fig(fig, f"fig_band_subset_{task}")


def fig_efficiency(eff: pd.DataFrame) -> None:
    devs = [d for d in ("cpu", "cuda") if f"latency_ms_{d}" in eff]
    fig, axes = plt.subplots(1, 2, figsize=(6.6, 2.6))
    x, width = np.arange(len(eff)), 0.8 / (len(devs) + 1)
    for i, d in enumerate(devs):
        axes[0].bar(x + i * width, eff[f"latency_ms_{d}"], width, label=f"{d} FP32", color=PALETTE[i])
    if "latency_ms_cpu_int8" in eff:
        axes[0].bar(x + len(devs) * width, eff["latency_ms_cpu_int8"], width, label="cpu INT8", color=PALETTE[2])
    axes[0].set_yscale("log")
    axes[0].set_ylabel("Latency per cube (ms, batch 1)")
    axes[0].set_xticks(x + width, eff["model"], rotation=30, ha="right")
    axes[0].legend(frameon=False, fontsize=6)
    for i, d in enumerate(devs):
        axes[1].bar(x + i * width, eff[f"joules_per_cube_{d}"], width, label=d, color=PALETTE[i])
    axes[1].set_yscale("log")
    axes[1].set_ylabel("Energy per cube (J, estimated)")
    axes[1].set_xticks(x + width / 2, eff["model"], rotation=30, ha="right")
    axes[1].legend(frameon=False, fontsize=6)
    fig.tight_layout()
    save_fig(fig, "fig_efficiency")


def fig_mean_ranks(report: dict) -> None:
    if "mean_ranks" not in report:
        return
    ranks = pd.Series(report["mean_ranks"]).sort_values()
    fig, ax = plt.subplots(figsize=(4.2, 2.2))
    ax.barh(ranks.index, ranks.values, color=[COLOUR[m] for m in ranks.index])
    ax.invert_yaxis()
    ax.set_xlabel(f"Mean rank over {report['n_blocks']} configurations (lower is better)")
    save_fig(fig, "fig_mean_ranks")


fig_loco_vs_within(main_df)
fig_camera_drop(DROP)
fig_band_subset(leaf_df, "traits", "MSE", "Standardised MSE")
fig_band_subset(leaf_df, "cultivar", "macroF1", "Macro-F1")
fig_efficiency(EFFICIENCY)
fig_mean_ranks(STATS["macroF1"])

## 16. Reproducibility appendix

Exports the environment, the full configuration and the split index (file names and specimen IDs only —
no image data), so that reviewers can re-create every fold.

In [ ]:
def export_split_index() -> pd.DataFrame:
    rows = []
    for fruit in CFG["fruits"]:
        for task in CFG["tasks"]:
            if DEEPHS[(DEEPHS["fruit"] == fruit)][task].notna().sum() == 0:
                continue
            for seed in SEEDS:
                for fold in loco_folds(DEEPHS, fruit, task, seed):
                    for part in ("loco_train", "loco_val", "within_train", "within_val", "test"):
                        for _, r in fold[part].iterrows():
                            rows.append(dict(fruit=fruit, task=task, seed=seed, held_out=fold["held_out"],
                                             tier=fold["tier"], part=part, file=Path(r["path"]).name,
                                             fruit_id=r["fruit_id"], camera=r["camera"]))
    index = pd.DataFrame(rows)
    index.to_csv(OUT_DIR / "split_index_deephs.csv", index=False)
    return index


SPLIT_INDEX = export_split_index()
(OUT_DIR / "config.json").write_text(json.dumps(
    dict(RUN_MODE=RUN_MODE, DATA_SOURCE=DATA_SOURCE, SEEDS=SEEDS, CFG=CFG, FIELD_CANDIDATES=FIELD_CANDIDATES,
         HLEAF_SCHEMA=HLEAF_SCHEMA, CAMERA_GRIDS=CAMERA_GRIDS, MAIN_MODELS=MAIN_MODELS,
         ABLATION_MODELS=ABLATION_MODELS), indent=2, default=str))
_freeze = subprocess.run([sys.executable, "-m", "pip", "freeze"], capture_output=True, text=True)
(OUT_DIR / "requirements.lock").write_text(_freeze.stdout)
REFERENCE_NUMBERS.to_csv(OUT_DIR / "reference_numbers_literature.csv", index=False)

print(f"Split index: {len(SPLIT_INDEX)} rows | results: {len(LOG.rows)} rows | data: {DATA_SOURCE}")
print("Files written to", OUT_DIR)
for _f in sorted(OUT_DIR.rglob("*")):
    if _f.is_file():
        print("  ", _f.relative_to(OUT_DIR))

## Notes for the manuscript

**State these explicitly in the paper.**
1. Splits are specimen-disjoint (fruit) or plot-disjoint (wheat); the published DeepHS splits are not, so
   literature accuracies (`reference_numbers_literature.csv`) are context only.
2. Leave-one-camera-out results are reported in two tiers: spectral overlap (FX10 ↔ Corning) and spectral
   extrapolation (any fold involving the RedEye).
3. Baselines named "-Lite" or "(re-implemented)" are not the official implementations.
4. FWHM is approximated by the band spacing where the header lacks it; HyperLeaf band centres are
   approximated by a uniform 400–1000 nm grid.
5. The HyperLeaf plot split follows the 24/4/8 proportion with randomly drawn plots per seed.
6. Energy figures are CodeCarbon estimates; give the `energy_source` and the hardware table.
7. DeepHS Fruit states no licence: release code and `split_index_deephs.csv`, not the cubes.

**Sources**

[1] DeepHS Fruit v2 dataset index, University of Tübingen: https://cogsys.cs.uni-tuebingen.de/webprojects/DeepHS-Fruit-2023-Datasets/
[2] DeepHS Fruit code: https://github.com/cogsys-tuebingen/deephs_fruit
[3] HSI benchmark and camera definitions: https://github.com/cogsys-tuebingen/hsi_benchmark
[4] Wavelength-aware 2D convolutions (HyveConv), arXiv:2209.03136; code: https://github.com/cogsys-tuebingen/hyve_conv
[5] Hyperspectral benchmark paper, arXiv:2309.11122
[6] Fruit-HSNet, arXiv:2608.01202
[7] HyperLeaf2024, Laprade et al., CVPR Workshops (FGVC11) 2024; competition: https://www.kaggle.com/competitions/HyperLeaf2024
[8] Spectral Python: https://www.spectralpython.net
[9] CodeCarbon power estimation: https://docs.codecarbon.io
[10] scikit-posthocs: https://scikit-posthocs.readthedocs.io